# Final Experiment: Expanded Data With Frozen MERT

This is the final Task 1 model experiment.

The evidence from the earlier runs was:

- optimized frozen MERT reached 36% on 25 recordings;
- a matched frozen control reached 24%;
- training MERT's final transformer layer reduced matched accuracy to 16%;
- embedding clusters remained strongly overlapping.

Therefore, this notebook does **not** unfreeze more layers. Instead, it uses
all available recordings, up to seven per class, for the five genuine ragas
with at least five recordings. The verified class counts are 7, 7, 7, 5 and
5 recordings. That is 31
independent recordings instead of 25. MERT stays completely
frozen. Every recording is tested exactly once using five track-wise folds.


## 1. Kaggle Settings

Enable a **P100 or T4 GPU** and turn **Internet on**.

Expected runtime is usually under one hour because the model only extracts
embeddings; classifier training is small. Clips and embeddings are
checkpointed, so rerunning the long cell reuses completed work.


In [ ]:
import os
import shutil
import socket
import subprocess
import sys
from pathlib import Path

import torch

assert Path('/kaggle/working').exists(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), (
    'GPU is disabled. Select P100 or T4 in Notebook options.'
)
try:
    socket.create_connection(('huggingface.co', 443), timeout=10).close()
except OSError as exc:
    raise RuntimeError('Enable Internet in Kaggle options.') from exc

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('Free disk:', round(shutil.disk_usage('/kaggle/working').free / 1e9, 1), 'GB')


## 2. Find Saraga Carnatic

Attach the compact Kaggle **Saraga Carnatic Music Dataset**, or allow this
cell to download it once.


In [ ]:
OUTPUT_DIR = Path('/kaggle/working/mert_final_expanded')
FALLBACK_DATA = Path('/kaggle/working/saraga_kaggle')
AUDIO_SUFFIXES = {'.mp3', '.wav', '.flac', '.m4a', '.ogg', '.aac'}

def counts(root):
    audio = metadata = 0
    if not root.exists():
        return audio, metadata
    for current, _, files in os.walk(root, followlinks=True):
        for name in files:
            suffix = Path(name).suffix.lower()
            audio += suffix in AUDIO_SUFFIXES
            metadata += suffix == '.json'
    return audio, metadata

def attached_saraga():
    candidates = []
    root = Path('/kaggle/input')
    if not root.exists():
        return None
    for child in root.iterdir():
        if child.is_dir():
            audio, metadata = counts(child)
            if audio >= 190 and metadata >= 190:
                candidates.append(
                    (abs(audio - 197) + abs(metadata - 197), child, audio, metadata)
                )
    return min(candidates, default=None, key=lambda row: row[0])

attached = attached_saraga()
if attached:
    _, KAGGLE_DATA_ROOT, audio_count, metadata_count = attached
    print('Using attached dataset:', KAGGLE_DATA_ROOT)
else:
    audio_count, metadata_count = counts(FALLBACK_DATA)
    if audio_count < 190 or metadata_count < 190:
        print('Downloading compact Saraga dataset...')
        shutil.rmtree(FALLBACK_DATA, ignore_errors=True)
        FALLBACK_DATA.mkdir(parents=True)
        subprocess.run(
            [
                'kaggle', 'datasets', 'download',
                '-d', 'desolationofsmaug/saraga-carnatic-music-dataset',
                '-p', str(FALLBACK_DATA), '--unzip',
            ],
            check=True,
        )
        audio_count, metadata_count = counts(FALLBACK_DATA)
    KAGGLE_DATA_ROOT = FALLBACK_DATA

assert audio_count >= 190 and metadata_count >= 190
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Saraga root:', KAGGLE_DATA_ROOT)
print('Audio:', audio_count, '| metadata:', metadata_count)
print('Output:', OUTPUT_DIR)


## 3. Install And Load

All code is embedded and checksum-verified. No repository is cloned.


In [ ]:
!pip install --no-cache-dir -q "transformers==4.41.0" "librosa>=0.10" soundfile pandas scikit-learn matplotlib seaborn tqdm mirdata joblib nnAudio umap-learn

import base64
import gzip
import hashlib

RUNNER_DIR = Path('/kaggle/working/mert_final_runner')
RUNNER_DIR.mkdir(parents=True, exist_ok=True)
embedded_files = {'10_balanced_benchmark.py': {'sha256': 'd813d1c89126d9a722781211c3f851423783035c3c11e21f0b6567907fccc1c9', 'payload': 'H4sIALS4K2oC/819a3PbRpbod/0KLOaDQQeEJDvOZjjDqXJsOZO7duKynd2qq7CwIAmKGIEABwD1sML/fs+j32hQtCd7a1MzFgF0n+4+ffq8+vTpP/3b6a5tTudFdZpXN8H2vlvX1fOTMAx/yMqsWuTLOCjz7Dq7ysdttsqDdxcfPgU3bfBqV3a7JqfHeV4t1pusuQ5WdRN8zJrsKktOTj6tizZoF02x7QL4VVRdXi3zJRXKgld1mc1P/yO7uirz4NO3QbOrkuCnLrjO820bdOs8yO+2eVNs8qo7aTdZWQZ5Ve+u1qL+tqn/kS+6cZuX8Keoq6DL2uvgdl0APPi43C2K6irY5FkFf1e7Mqh33XbXtZOTk3EwF6MLsK+BggFfuiYDZNxkZbHM8NVpl7dd0G7Logvm90HdFFdFlZVYbnEN5fPNPF8uoYk2WDX1JsCOnj8Pyuw+b9qgXgXzulsHf37xLtjUy7xsoQp9G98WbR6URZVnDfZ3nuMn6AsP9a7LG2ymyncN/FmUWdsWqyJvgtsC4EGl8j5ou3q7haah4qIstuMyv8nLIKuW3DvxDIXG56f473NAR9cUC2xpky2aOnhzHgeLulrtWsTgJsOveRsH3fjjzxdx0Bblut7lXZcT1NfZTZG34x/qXQkjhrmtG9Hrq7zKm6wDhL4DOljWt1XQ5Nu66aheFuCbss6W2RxmByoB9QSfiy0S2skJ4S1NVzukqDQNio2oWdUdzUF7ciLfNVfbrGlz+Xy1kL/WWbsui7l8/EcLsyl+16381UBv6o18ate7rijl064qFjBDMOuZfAU9XAE5cQcXdSmopJU9fFXvgKibOFjmqwyGtCwWHRfeZh32RhZ8D4/8obvHCZPvX1b3amj/qOdG/+FnU7eqJzAz27Lu4G2yvcdfQdYG27JT34vG7Hi122zvsUi1la+2MHR4gdWWCgF5Nq+bCl+2lUJSC6Na4rjp/Uq+7upmsbYekoqqVpUYGr1DlLYJdkYO8jX8fgtzj4j6lFdt3eCbNheoaq+BwTRVwkshpVUiq76tr4q2KxYf8isgGqRRu84GFveqLpey/CegWqcEE7wsEJ0E8N8rSfHvkODvXhewvLP7mL5liwWsuMV9StTN7yS3SH0f5dJcEKmmTPbik2wnpZV1x2+XtIrSOa8iE9Tq3HzSi898C+s4ve51ZGSPeltsc8Snoj/x7JSCvjb1AhGrSfJtNs/LiwqXAszXxw7Jpll+XGRl3oh5/udyI0vjb/EW1lYLrHmDbE+S966r3+QZLuuLO2RJQCIxvX2Hs3xycvLul9cXbz8G0+CBBhdC7S7984tNOIHf42y8PUUJM745HwMHDRkD4YJlj1m26nbZuC23p4ZcElX2Jx9fvnv/9iL98PLTBbT07Nuzs7OTDy9/fv3Lu/TjxcVrePftM+gLLOKAmEsKXKaNRsH4b4rfJD9nm7zdZot8Qn2glw3UVAVeNlc7lFXv6Uu0zFnyweRP03RZL9J0ZNRMsuUSm6EqUTgeL3lJhIqXTMOWBGm6gKkC0lrAp8W6RvY8vfR8k6/WRbXctTBtRTh7tMXxut7kZpv48rTJbk8FNCXaw2FY9IEmZjy+JnnOsJu67sSUEdmLNn6uq1y/Xefldhq+7LpssQbpIfQB1iEChICLqAORjCQKbG2eN6RClEukM5KF/+fjLz+TmHn3/nmAfKsVrR4cPmsD42XRmONX402FthAeBAIdGmNPoUXk7PkUlBwN7cXBuiSi2zHoOATiSyGYWM/vFuVumQs46ktG0moaZtstaF6eqbicuRPxAdFeIgsARhMIuH9B8bNYE5O4qlDmI5eEToCisN1VC1h7pIDhHCyAio9Bf5tf4U8eP6HCi4BvjwECf4FIlmoSVqBpGDCenyVnB8HMcXTjtvice/vw7GDldZ4tx+tiucwrIKaNH8KL7x6HsWzqLRDdwCDOkiO6UTYDtc/z8fPHq4PkWqz9pHx+dvZ4fVB6Clg/fiyeH+5/ew36q9QSAYCkXVBxQSfsml1+eCWC5FnkY9SC26+urVX5QyCaHARMJSGZAkPIEFQ90yVoYBFqgRNS/uIALIpdPkGNjwQLckEpSro1goGuJJtr4EcRP7TTT9BoDIsQVKC0vqZH7gKxPapXw8qOwlvob44iG/o+DXfdavx9OELVbA1LshTt4H/YtYS6Rt2JRYEYjDOgXyBUhNOiFp61i6KYvsnKNpfj0vZYiopNcRXhsCceGUkjRGX4su1Q4Ff3s4mJuwfNvKTcIzCsN8KjZkshaKMwN6QWoqBnAZMWFTDnMChWYHmAjdA11JdYfUc4KQsgQvUoANMrB52C9WSDF4bAw1Pm4aIP6oVRiJl1CgigT7Ko89qoIFjnUoEGtRfMo8jpriiWCglwORuNzLELJkkNMJMUDfe/9Kulki3adeRrs0K22ZbYiQ5xbKhLRhk2naByvoQyhvpklGELFz6zWsdf9oJ8mGbSFdBo3mwbYA4Rv5o4pELUA09MMkTXIPKnmnhbUTEmpKbX+b1cK20OlJjBim2nURijVjQJR6OEYURyZZikKMzGpF1nwKUj0dooWed3y+IKTP9odDk5fzYTgwDpB/Y5SIplSlIy6sBSn2Bv7V4vAcegA7fUccOyTBSAKPz5zX+8RlYJEKwuhWHyj7qotIRfrLOGvB70AwxvDT1BeYvaUDRSxWFRYMGkaLMSaBkUWRTMYEtbHQEIc9KqIiw8EhKbRwlGU0YrqIIVHTmcC8YY/G7wLzDhPwD3ByN/vQN7bNzkwsxnNYKMA9bnnoCtCBgFnMgGQF3Ly2WboBtAdJ1aQ3+RbsHADb6UJQswW1DRXeSSm+E09KrQtwQ+FVtABWDxMAykRQMIoh0IDLEehYgOJKqu6Er6QUPEH03GC5h+hEZ9c1QJLP4IgDmfjb5CL7kfl1BsNpK9tnojUFjIkdG/IH1soCs04oH27KnkuiO3d1R4sFP0dRhhJYgnB2FFl29UB4/pF1b46l6ZpMH0SwyReGpEP4l4YWpZuFCT1pq16FnR5lTJFmawaJ3yJylUJFJwbRmIkcV6xOQSMXIuaCYyycckI4RtdjqYToOe6ceSjWG0qq6ExX00qBjb1D1iqM5syCdFrxZvIeydeGYEPwxMiPCdsaRqIzGmg9OCZHWpBQP8M5NKRH2LWLucqaHxfBfLmH/hMNExIdtJUKeUbY8SJDb4i6ppu5gS80LTRticao6Neeuae2fEWHLaozN7LDZBZ7tlUaeotHkIS3+0SMvAOrVIvlQNBx/rNiFFkLTENtJffUwGEJewMRj1PuJ/D963WgECFIc0Q5F8NLSVXhWhKNmKUa+UMfSJGk02b/GvORw/iH3vrcYc6Fb5tgsu6A/Zpy2+s/HCisgq/Aj2x5Y3DKRoAsJ6kOPcT4IHqLu3lQdEqEPhQv80CJ3UULYCHiFsKUlx8wMFPJggjkNkUW/vSfsHQ1H6fEGqrnZlCToemOg3uSlLkS+pLkgSMflRhnsQH3ZVB2r9RdPUDWBiwBcDhkmO/rv7YFnnLYEmgIAY1QSix7NEFXfBiXSWLlaLgzRmxw2uXKCB26y81riL0d1T1rdlUV2zoueIG6xKjAxqE5ie+JUlgBPc5k2EGuGyRTxGoGuBWhl6Vovda7lucBoj7NUoOFVgR55hJqibRgIf2Evrs2JRdiWXJ2X+uTiKR5GRaIFna/EoQ9FFglS/kZNGXHjk44hKtHn1pwEx5+FUDO1oseQwyomXV8xBM712KQPp2F8F/Y5FtctPPGx8ARjA/cEcqVlYdj0AiPLeS8SHJAB7coQLAKRTg04A7+CoYNGmSHhCr6dX7W61Ku4kdSPshzDZbJ8jjmE13dDfVZmRozjZfJuFNuMc+dDiDvVYFOV3wLjSFnVAWOpfjha33UFEcBNTB4/4tldFgE+kFrUKH/q19ogyQptVfzYsx/VQL89m2C9j7KSWuWOBYo7qVIISd2PpYNQf+SHtas0JRwku6pRnOwpDZwkeEu9+0W6J9WyV80KTbSdZm4KVCU2NBsTv4zLelu+oOugXMMi2Lm9yL/ghAv06qa4YqWJo9vwb8t0UnojSg+LS5nXhzzVbvdAQ7kqc4o4Eq6TbrMD9irzJhQkE/88bU3YmgU12QhCTNiDYv9AugydCDrwSNkDwbtfCv2Jb9QnzhRpQ4ILsb9Cgm7/o2kCaE1rOJ7rukNIjlB3kRukC98L9Go9WdqBLQtfh4rjxJyiEN24mwZlhZ+mXeyVG0/9hbYEXF3SMJL2S8A6LdfmlqHUM4/WxUcTFpYWIWfDNNDi3+WqpG0JDkDWXYXAOFg2IYia5oFRg13Xd5qna6hYarFLnJl7Fldet8phOcIbFPrXtGjW+2K5RARehCTNcqBHxiUM8vvYFNV019W6bLyePlQbQRqhGRK4Lq09IkD0fHxEA+14E3UT2GFC8XM5Ge0OrvcViNvsoPO5DKHHJTHQ2YjdCpfpiz6wY4qVRYya5PbwTfAto5KpA5xtI3oi4spgI7r/5BpsSQKVJjF0Eeo9klb9N3WmcWa2wjguq2LTMNvNlRo6fSRCNEQj+vjyfgZ1dsHyUa0fobRx0RSiX8C4nipJmkg0jKFl0FPzVoLUTm+C1JhbRjMWB7EU70rPHvfENHjraG8lYDuILZMAvVXkfPFjd3gdMJusMJH3WYURd2wnjUuN2L5DtCoIVmAPL3SIPjP1mpLjeFnK/4oUka6K2loVcj3BB9HyqtwKNUEYwkMn5i9neFALMC0CaTkUgVfKB/kTGbgBjijfPcTEZa8KhPYkd04Shb2oe+dkzLTQrWncxpocbviRSxW5WVwlvbihgDtIt0ca1BUO8baCZdNvkK6DOdScCeyJjfCnISrG9x6FDA9thIpyHN7Iklm2hGB/FZxXKHuNzojgGLbZHFHZ2JNm/KBeVCHIzWZXL4xiJy6K9xsmjqLoEn9Jdm13lkcaW3IS9RztSRf2Y+3xS4lvYMraYACIwilRoWUJKQXlcb9QVo2xWljTHXErAChkfkTHIZFO3uKe52dSVpYv2d/FoI83ctLO23Xh2VOEH9mhqNuTjQrKWZEN7EyJOoB6jrcPTx4lHsQ+7usvK0Gy4r16HxlYm4UNOM68tY56lM19wUhvUvmfsU7fM8RERytGp4nsLy1tGHG/fDwzUQOPToU3QP6I3RLhX8143FFob1N8jLJbQq+A0OM//HAfPHNwArdJeqVEe3wwWXzV5bhfHN77ie7mzKvf0OdxArzOoFIrIxxB/Ky7GOmMsl6Dg6Wwuhb9VYfBNEE5DQPB334/MT+8/XLx5+9OPf/80UUYIGhkcFn2TNyr+MbSqeUBpXTz8SGtcKOHBg7XkL58Y2vCT2Z4t6diQcGDCOzVshRcrKUeZGZAkbMNfW71LmYuIaeyG4iT70K7QF6Y9bkDy9EkFnBRat1GBElb4cEkAGDszLBnpPXHUAc50/sKwZGSnwLIn3UGKbafVj1LJYpWp1+ggGzrYkl6KUmdxxkomrxA/RpuHV6NuEogfaHe+I/teS6BHWJPhIjD5CQD4ItbhkCmTGgyeOusffYx6ldnmXvWBAvPbuKeZPRDTNQc62rsGt8T8G2QDt3VzjZ4MZAzoc7f5wyR5ttoHP/7gXX64qH/DlcmKjSlVoqNViuONsTAMf9xloCF20LscmMO92L0CYyVraOLxiANPPscmYCnBTDLGqdrGEMXQUUBHM4BDXs4AnzfEi+knntCg348rm31ac0yevwbfult9qPD/J+5kyx0SXgfQ63zZanV+BcxItshHVLo1zM/4NrvnQSSGv45HdUmjUGYcV74cP5v1C9IYeyXPfSUJS7MEj5DoohOCqjGaSu2ZUMuKygOpeFqtdtW8vaG/0A5PbwnzBGRtmzed247uWdEC9WKISxf1yhBCRv8yGEbXYTiM+2OhyAgNGq9cScplOhAG5ET0cPxOLzzHiOwJx6nY0A/hhxnzw2FCol0RyVWvViCJ22i5a0j4TgIR9+mEeqn3JCLIFUP9rLYJRvc3mdg6Esp122UNGm+bAoVOchYHm+wuOsMfsiXgK/D8Qpjyol7OkR1Q1gRkVBq7HYM3uhE/bECXAf6vU6uTvdAKGBGNJ7oEEPbMwaeyqMggc/qn4QsMSTzjAYmsyZmnR19mU/HREKjnWokDc2N+M8SS4TejINVJMK/r8jHfGIbRSXZMZ2SAkOWeheoUKYniY8uKocS4VUdtHKs4Mu7Jib3TaFX5sp1GMUP9HUbZH9Vnuy9OP4Tl2Ww64LuRqjOy5+LYIFuFmsljeP4S4cTeiDZF4UG+L2Q8rPK7a+OpGYkpVsMqK8pdk4tNdFuxSpkzG0JvQL3i0kgBNjVoKP2ix2LN8YDSHrdUkmhvexW+anJY56DKPOgG99wRdy+4t6lN2qHkE1NeOpE4K4f7wqn8SLHXUxJoxg7UzLNnrDbnWOcWHKy/EmE+nseeFdqHKNgy+kIGGHWPQ8duN/pQ2+yGCObMu3nKLdCOCLfl3wMXVlSKSrHAGi24wficHgaHQ3naZuoNHe7FFNRVzQQ0WITHwKcYIn44EGkksTodjHAe3ut21EAaKWqB9jqdDLZNFQCbIF22gEeB3wjEmbPSxwb8kb8PKPcfb4n+Xk5s8LOhQTEKoXdI1Cgd563qBAwTpOT5cJP+/X1FdnIjjbbTje1jIJknUot6Mhvt0/aBiHdy9my5p10xLzwhnzR30gE23vLtKiEfLS31WJK2yTO91RRfv9TsZ3YwLO5waBx70/T+9qM720Mb8I72/UhFufOuvbKPVOBlZBwKOHJ17b9gFTGP6u1cmhulzMWmPlVnsAv9yB0L2l+PBibF57863UQ5ON9a7v5vmWYr3uILuDefrCVtgAIUKCIEsHtUjSb/565oyNt56GTK15DWUfEeR8/tgQjXL5nTr5rPL5rLr57HMEcvheBFgKhDEbr2dJ89WvIrptkXoCu1ejlj9vk09mVbVgo+pGp+hQtbPo8Mz11+U+S3IEPQ3eUcolF+PPIiPqFZRhfxqXxDJ1TMF1qE7Sd9/x2XMTAIxU6ttxJbCJS12972hHRgyrFcTs6NALHjd53DjzXIYXaHSwNggfkFyFzbNvVNscxl/pSqrsYtCtYueH4mzs5yB3shSK/zDnom4paY2XZ10J+O5LeqhyAxG/0dZD3Jls0Ya8ls2e3qrTTKy7pLZZx9u9tssuY+Ot5YiwN9KtPd9+wdLHAd1tpc7hlVPX/0ZDA478HmNbHiDKaDey8MvoY1rO0ywfCuN43QsASOtmWXrIormIgI/uAR5ml0fh4H0nvSVm1C9gRijYKkpgQxDu6mMgp3vcunokdxUDfLvJmKDXgqKjuU7Krin7tcxW9g03ddgU7kRuRpmT5/gUdKATLuNYW6HB2YilROIRnHrE7bT5j+AmAm+qQDd8kEgtvwZXZf77po9LVnZxESUjKeX2WyW26L6fl3Z/rzoqzbXB3npcMjzHvpoDj7/ide6pJaPx2l1H42KqqdbcIxc4u9qJuNQ3MDRDRkuTHNCKmANpjRg1jbWpqXqGYlORLkJGvxzDaaCKQXPn82slahqqUOA1M+j1SfmWaOREdA8UgmCSCWCeuVPtsTf6FvRTjSMHjTdaTRbKRI80c6yXozoMAeSzy8wzrxw9ThLpuCUqqkau/CZSmpEwRwgK9wH6XvTnYYbaPwQSF7n5q+lKTafg6t46AKxqO+PHuHFL00Cw6c1RONW40K4N4JQhalySAm+tRFY9z2qW/TLfCMMneo0gjeQWNSD8agsHAmUbC8DM3XR0DhPVsLgnx1RG2xyWnWlq9Ohq13mwzkcot6FBAbEzSyA5BtED13twxY4iPHN8UCRQXnYeLHKFzslhkdlOfX+IgnCLIbkN/o9o7k0fjFdifmcomcQAEidnD+HZ1FJJgJf54GAjZVN8o+f2btXv5WYfIn8vcJRrAPQI1/YGD7IHqg9vYj0brl88tl2iDoji+bUILHizFIi9yvILJEE+gG3bUYtLWpyaW2dAmO80xNdTaiLwUV85hT6v2U/lWxpIctlsdjCF8p3U2mxMBNIvqrkJgEP1UiMByzNlAww0+4T16xHzDsxaZ3ObDFa1Ddu47y1SELaHJKvPeLTGtH1oMMhteaGx/jhpeaySf5TVZGowQPKNBUjk6GmZxJ4sTwBuiaUuOxnM2ljHWlpxMnbGlgvHUlHc9NVl3l6IszIqEMsTFSLmmDk1peadcfzXWZbU95ZJfc5EQ0/Y0B3+Ysprjv6RQG4FgTPccuIp5QmjtB55gBo6UzKKJ03+JVTcYeb7IBeTrQYuw5vEM8d8B9y1wp7SjJWzsNt2b+J885IxxCyufXaScER3QZmq9B7VT0NeU/cdBbcUpH6qDpjvOtUSwig6QDY6H9MeyF8DuVCz7jaGchGGzIftFbEwf3M2j7jPlnUa3A0qpEvpNo5PdiMbEqAniwMTax8ArGhQgL4xxFuOfZUSmcwv2Qm+5LkOHr1qWL7lkPSV44VidxKwZhRk+fmrBHiVXK5y0yZN0vu+6X1Tvg3c09sdpB/5FRJ99su/uUOJbnHN5j7FtZzz++/zXYUMuwErHVdbaj3A3LXYPyUFvfSpmRqxmQ5BrjCuwHZNlMNWbuquD8L3RquMyxCQ6moSFgmrSyBGML1icGICZ9uAZ/d5eFPSMiLwdJI+Cp1sdR8G/T4Pz55KsRBnxYaQjMSmAg4VDRs+HekUbS799eFA/4+S9DCO4wca1OasuhXC0MzYe5g/jCCB1MLbPN8Rfi59+/+/4PRpAY0xIzM1EW14cDXdh/6aChv95Ru8O2BHc/fYtXvqNhdMlRRinKbKvfs56zjOsWy7tYjhkPy1S7DWWedea63/QWTEKyTLgg66qgv2CG4AiwNz0fJaADwxvKngp/+ybx4fFcqh6q/SRu1EDXcajyHsz/uTaTDQtGIfIpGwrMqa3AOFv5BgSy0QBdi2ub2lDXAq60ADxWiNjFelfh9n0GpuP0bCSikvAdToA1GscvzTVij0OTO8MWGHdEhO0cjvGcSeFfzzH5swtPHSBx4TkxbcfBROyAAvUZ42+3mH7WPeRtmLa2EFI4nrpItwvy+KcmMmJHX8ABTc3R+bCpjruSh9YrBAaN9qOMbncYJ0cb2+bYTo42ss0BS+u2ZGWAz/stEpFwI1JRYoP2rXHA6JCI950B2mTXeASonuecDlbm0bWi++RLTRz2ofcobDF7buhm0+1tDveZfqjzfod9dbufHdkvNl5Nz5OBzR2Cn97m6FqdhvLwZxgP7JzfoYnSTJ+9OBuAR7vezTQs56ur1gPGGbHxOFMZyKhPYD5zBvRXCgNRVSVgqe9KeaIXZygFpRBIJcVA5lUsVF/g5uwSlCJFv8Djg9RALg6rBiLvptgPN8OkdlvKFaKaMNgNNJaAnY1B2chuquRj/s8dKrdgElsjhE9vkUP+DGZYpHrnoAELUfZrXcLser/0jxdvf436r1/zUCIxpMFWNGgLIaPY3apBDAPDvAWaNREMqOOlxOm8aWmYL3p+KhNfkdDidYwlelPJKuATK4xAechCG/6xcCvJ8t6P9TybF8A8itzzuTf5roeYSEA4hynRtoiCfW7jaSCXpUzKrWIr7CzdkeDw5hCs02a9NOMK0EAC8kchUpL/dHWuAMkM476aIOFh7WZX+ZTrAcP6nEP1ZXFTIHMB2W/CRvw873fVll+eZOVG08ZkxcH1lErHUiqSDCcHjol6vQZHdnpJFvEWzOgYmhiiMxUG3v/ESSKbiZ0rnWmp24HsNbxVnl0Hce6Y+mpsDsjXSinyJDijqE7em2vzTmXhArToJSc8EkZ8v344cU6Tcx+Usmpi6hLhzFg7Fprf6MRSXLTnhrvMNXpN8EfZhsBdotLFR5fyeI8JdGSffBpdns3gfzP86wZ1swZrDWgUaw3Q7sZIU0tRpfKWjVSL2eFdLK5y56UU+HDf/wBD8ZXH1/fHMKfHzxTz1m8vxJzzsJpvDdrsC1ZXMeltxTFbnJn5O0ilQa+8q9MYKKEyaHFx4WTFOp2Yd4HPQTML0eRC0LUJt966RgfkUcWpfe9E5CiE5Phno8/suMHsBsrdG+0Dar77VjJIHoG4FIU2LxjCj/IN2p1ZtcMh5rCYewfoGX5Jt2ZAbX2FRmSeltdD1F3V7ucpHa34Lpbnm3Rpk4+3691qJbbj9FvV9an6JUf2B282ccmDyPhi5d6AlmZl2YcoN388SuaJ7Z1Gx4CgVOFFOTeT5itFakrHCjENempoV9qW1It7avw2UvKzymaAEW8E2k2XMplQ267YwCRr2qI3yctltvmvyN7lwggJ4Bsw1NZUGsvGaK1s9Ac2B9JlvsjupxghbBL1AkNuGw6/B43yVVO37UXVQWfv38LPSC9e3gbQ+4h69eil3d8+WhctZRlU8m+OAUKw7q/N/R56Sa6e3lvKYq/i5KEMHu2hzPb0UslULkfnbVHJOI8DjQ1R/pvg3BSshEsiBsMUKGHQGIaEwJMzZ3Ua2pUekLG+HVEvu8Zr+E7uVVH2PJMj+DalMGWR+OXSil3wXhW8HyqoiCshBfCqyUjfSLs6xbPNnq37EkzRTjvsRUfcMi2WUAQUcSU1yn7pZA6inYyP0UnfVQ/Ux5cNUTwadjLF/Dpp5CF6sl3xK5rCQ4Ntu3wb9XtBs/uNOmyC/ULHEZjvwVPird7+9whAHsjkUYN2coUh+cLxuMwx11fkeCB9EIU6JYDJiRyq7xAvb9mefPHOEy5YZ4r1Ch/1itpkrys7g7b6euKsDWk04L6RbT+YeIj7eNbdwYaHwZAaFrv9tWtL+wmPOJwfrHeE8WRIDWJx3kBlTzoPYkZgX9FfTyYPyRlayoMiCfbUI/YHK/esOPu1r6KJXFXNfDlUqWeTmi8H04sY9CFkgVUvtiY7DvguEytvtZYjxiYWPf9Nf3M4qyF68E//oxQ29Lf/WUqo/qxyQBxncHcW76KEznn2HfXha5EjvxKrmlpJKaHAqJfapB+E3ROJwxFMVtHesQ48gGEW+NvUEKHyMpj+Rgyf5jgxp4Xx1E/6398GCT/RzZDidkTjWsRlwQEz0vvNiQT43ijqnzx+zzijCAwDcbobZiktHyXrHA7KFat6JA0z3DAh65giTndzjMBtUc14hukLZawuPHybyGhdrICJ0ihYVwTf8uoH20s8G6t9JnwlU35Jaz+0IWV3N+Sa1sSKR5xL0OvDecmXLuF3sGLABAjHY7z4ptyuM1D7XtiQUP7fcdK68IL61P9+L76TUjjOWSv0dYuUCY4J/iQnyS12fgweFAdycaE+HA/R4mganr4K9GuBKo7nBSov4nSA/kEzd/7IzJ3bM/cRRZz3e7HBQCrQnp7ZX8v8CoWYInsg9S1PrRVWNfEuWrlCH4nyNkK2DY9HLwZWuXOIAUnYybbC6xCGo7zpTAJdvslXI7XCCaK9KI+0JJwTDMMya7HXWr47+aQ09wFRqPmPkxVKmaFQaNgQpaLa8JSZygbtUSpvGKJQwWuWckYscTOZCdSyTrUHWxGrHJKjtBiJto5Fb6KiyOxjIVg8DuRkCe4bG63KK3voBl2fZ5hATDxOgNhwcU1c59iJvcVrO/K8aTWUs6SCgXmsk5Hwn+hWvS4v3eagz+wohd5R5HvWed8yH7n7OYK661WH6ry04/xqvT60w6ECwEJhUTgeVhFG4E2AKE5W0MRSMbFxN3iOxyuedQPDJ2f+bApjLGBydgJwQFyENfLgR4XGMWBNgRG2frCO2ECwlsgguMeLjBeyjZXKY8hzEjxocHujMSlR/s5XH7IGx4PRhXxiRZ8HcgSEcS02UOPnvOJrsS1wrhTCt1dNsYzkQIz3lmQ6QrQcdRqICLlrAcsDLMC3oRTQeAWlD5LtNm9Ac70runuRg+J5TJuPz2DEERpzBgMIxsH5KDg9DZ7Lg16Sbbj+eMfrbsDgRjHzETIDvDpZi6qKgl6ga3gc5plm2rqPU/0zNnymRTcNtwvzgj26a1hFGofZrquNr/JaOSSfae9WObf3gjUNr/KH8A4kDw/qchIHZ7iC7q1X5zN94I7naj/MEb6Pg++Mo3TQftfhuPuH6e5gNd1PoS1xoE4crmunuLQkdX7vLgH6t3ee7nJm0Hz/lSDs+by+Q5dcVi3WsMYjXBUwvBGQYL2YhrstnqYr81UXft0SAJaLLRQVRnaBXu2c6bPXBZpHO9RnWKgM7qIh1gbOT33hltbh3bHBKygN+EdnAlJtH1tD7HICdZpbxLwYiXCUHScipbjaPSut6E/obZMeqgW4P1CH/G9uO8DTDzXDLN+sY6iHIqOirCzepkKV1oLW9uGbjnornFP74M0xmhFoQuE9m1lpKUE2EKPUIVr2xx6yTZBGrGYs8W77/tC5h+yFQAlPn4HRAWCGqlTf9jw/QtpPjFjWk8e9ao7jkuZbOyDdKDK/k025L53aXxb0sXcO6pinmikJveujo8QFrkqjXprq01ihpOe1G3DYYZhodR3O+j47OhRBHydUIfYgfn9iJgeUzchTDno7iZUgztPFrQqNTXJCR7G17RvkfrEHacZ+rVbW9B7QASPXaIstW3Nb9o4JmfmBE8aroxMH14Tuy4yvsvOQulHGsD/NSF+Ll3m7cik+xsy7ZtJ5UAk7zbuujc+0svVYY6MxYX1jJ3FHzGBnQ6PQFcSKNxuS654Bjsyy9dxfli1Nq0ZJYXeURlAGuRFn6UW+MWOOdXdi3ZoTgmYEKfFX/q22FAfjoawhGBvQfcbv/SijdPXHXAZA6cmXjRM+dU/k/g9Ry7mFGy50EDn2PQG9eGndZv+dOdCen2O4H5fkIrGGLTiBShkr44pkblVSUKV4fUQXN0lkUeLRoMbAgJkbvyjX9S7v6Hpm5uf6leDrRsuxEPBWuN4yuwESSOd4spQSFTIc+/XjsPY6MBSVRroJfEehx0OBmYa/z7ni03b2L3ALXYEDgQGIuFMrwtpqOzJCUDjhXRe8qdz/u1H8lWz7HTX9umi3wCGixQY9G/RbTPa0p/mwRe+cfZhmjnRfbLLtNPyBTunZX1jvT808GnZNtOLnWcMXsvuOAGR3hlPdTcViWtkPznTsiRloxAeM+Nhn/fuOHviNCtewsHt0QLz1upe6JGFIPcWehMEy0D/L0+tQrcFpQ2YB8gowbenbbPPuSJ7p4J2wbBwE8eH3OA0A+9QX/aYH25BKB93YLLcsv7UJRjL2QyBEOIKCIU7AG/7bWD8qZy27pXseWxYWhwIzLcVq6lGxhIt8epSO49S6n0q9xDIEJLBDqpBdAwGhXnJ8+BcashSN5SWBqfHb50Cf6p+mSBOeb1Nf8TrDH5suU07ZMDnTh9OMT8qLMo9rQGp0gvFrFDoND305QtTLCobyZL06TocabvYPVaWczloYP6hYWYWOw7kHC7H/o38Gjte2fJ07rG15+iYdczZH7xEKXjBjEHg40ifKrA//IyfLBo6KaNYhN8ocEzDUG3O4HdJ3tlhFLd0xnLjapFFWmKB5uyvpLiSfSSoFYe8GHHP5Usd6Zo3jirDmV9ewXscn3hNvfJ2MuHWLhuR53z/v5rINx+Lx5bPsYPGIK9v7ql2/uOMe6RdQ1/AsOk+ejFE8cAeSSaI9zD+6sSo2gEmkQhFLwh6cRA9nfmQSfWv3/88kDsuA/y1zubcujzPHp7d9jjnY8aUJ0sSCNku7G6juLW+0FWjd0Bz+KVAZ+t5dfPgU3LTBKwALqgc9/iCzAxiKamj+/oQ5BDA5xQ69xBmy2GJOp/PL+6Dd4K0v8x3dl3KdXeVjTBndSwWYmAAv6PYYNlCAQDG4CwBTrorMSncZdOuMHHjZvMXEllRExX9hoiO9dZoMdf9Pfwo+AqfeDnxfhWN5xe4k+G++/ElEVu7/2yn3ga825ULqysq9U+qjlbBTJj+U1dzrIZ3Kr1TGRBMRk2C3xRydDwPXHPmgBPr+EKuauqIiOV/tA3ktAeD52bfB9d8/O5AokOtUo/lUTZu8VUtn3n5CU4O5wU/dLwDA+x6h4QdzcgCDdMJ9V+EFcbXMYcXpaDkaEJq3a/wA3+awCPDiDdpl/gsUZHKvG4vc+RgEDLjJAamYmn9pg+LTtLxDMNH3mYIN7dmnj/EiZ7xxuBNleKvnG6OsDV2GpRAvF7E/Zk0jkwOFfWg443mG98xB33DZdTVdgX2A6D8IbWCgxO8B5UcLfg90hAw8/IDTy0bs70xGKtZaPMtTq70PEifyGbo4fg4PfG+YUZpf+ODwFwMQvxCQrO6Px2P5/8nR/wgI+gimtjUFs6VtK8acPxus4cmgasYz6a90kbCTKUBoXqCzmsQVuokCHHUBiitqYRkZsKZreETsXRIjBm8qRnFp93fmlIdpmhrVLm09YtZPW+EUt7UIuzyJosG83SugQDNB2++WD4tQSS/FMJ5odRozL//uSXizCh+w95dPJEE9mU2S5ysBGj/0DlzrEgfAyU2uHjj7rPQjsAhT3r6JL1/aOVHN0zvxZaB74Ykn58+COaQiGzDbxKtN3nTpn19swtmlpOOZSyaX+pC82BkGVmXC+iogQBpdhvQm+jYmsFrLkadlBhJ0hI47EXnihcrI9bFrdgsE65ZynzWf/Kic80G0BiECq4ySK2HsyAi+vyan+/gHdroHEV2MbRXpwbYZWNjPm/F1fEoYipoJ9IxJvVKHV6l3hRKYyyd6o2KISmn5iNL2doSHEmWQwVdO68c1KOQg8Cgh6SMTugrf4NH7dS7M1kA4VE0dYVNUu5a1B6TRcZnfAAUoOQVT+vTpA5Hn5BscytOnyWNkREo0arJ0lCFbdFo3jkHVow6JmW3XlCd0jrpfjvoJJbfDXO/VIhc6MELjPECg4yElgqZcAXDyESOHLkCXBnx3dCnjT9WygM/MXxdQYLODv7288HgIAw/oB8UGLZwMNG6gl3W9rDEWFOst1jXGvELj2F/iMwI5Iuf57bpYrCllPyqoYK2sKF61k6ml6hXVbDHIqwHVs6HFyKo/XSSJj/JcJGn4qCE+il2ggbfFpuB9lhZMo2XwvoaxYvbU1012OyfH02OEMQ74lnevnq67ayY0RyWyRW8UTKFYjqC+3OP9ftVVzipcZqCBZSgrlY7RIhTEl2YCwbtF3my74Kq4AQNpU0Nrm7yswQakS3xgoYh5r4LvnTox2WSZPjhD/YWO4x4lyhmMXwZYYGzUeE3nZgfzVtbQ6QZLFfWy9XTOOqZj6CTBOhP90xHQ3DV3odGcLuADtgpqsyBlsiKF2ZUEPwEWB8y8gGIkWmORLNb54hrzH3p7O5eGAc0V2wZiwRUt5cNrMMUepuTNOVAEIN8CHee0rFY76Bcsqnzc7ag3t9QsLJAGBqBZArCUZbYB81ecsRdLDP42OWYZgxaYNj3d/KDWAU5gIXZDxVShBFliCqIOA07rCqYeEVJULQiwjQRLGG07TAhZVKtyR5zCPK+J61FSY2/ZJgOCh1KLs88DaS3S928Qox7FQe8uQBlPCy0AMqRH0rjDWjhLYDYWa6BqZ+8Yv6nrZt8zELCN0ODZzSm7MN1y8RktJgLZCoOQ85NT7t1TbUKp62ZlayLhunWlttkT6Rs2y6vk6kbWfvPzrgJsXEu/ckXXRlsOGPsC7w8X73/58CnZmFanXUJsCw1+lzJ88DttG1mmDs3852KLN28l/7fYvoG/kTkKsDVuMUhMFvnpffr64s3bl58uXtP9jqLswMUYPOiJe4wQv6NvHUH6st4KoOK+LyzNeaWTJi+BrKFrXW2QDp6PSLd1W9y5Z6jz0mgNQzk9jWGHZZ5qbuSqrOdR+DQcyMdbrHgBHBiAfyCMT3Fh2HEDEVsJ5oyoXHWYl8D28aF8wj0fdDGm+CBz3VHUdeJPt1Ftk4Pfj8jYQbHSGI6wBqsgEgcOQOJ3OQbryx0YgxintKAjkqfm8NU9amx71BWQPOW8ltw45XdUU8WJgWiilIgioKS4So2XEb8aGTdANHXduYs9pE8hXUKn6to3iaoLGai+uKdI3PKjb2pwCxlbyG5Acq8PcoGzT5hXs7egXOlOvHK/HK/4E5fOzXN1sdmSuflvRt8izR4RDG3dBPDq19cvA7V9NgHzc2hjTR40OW7zTcL/8f2vNlC6EJSOMPHNhGcKMPEBn68TDzE/Pzs+V3/4MsA0z+j3Ftc9ccIu1Bm0eqa922bSfrwk19WsVZb+eivEO6uAII0/fYvKw/vzszPUJuAtp+Ux0vgPXSB/oZUXokWTpAFjxpPEj3FHk0F1SL3uypPXcT08fcovYiRbBRADzQzw+nYLQv41YYLyE9D6sK5lNx3smDo5bGmrIF1koFR2xSJ8NGduOB5zC2MEM6YVCGoY0ispCls0Xlq5BfFKwCU1wrys3e2lxa3cjy7P6l+oknUdKyFMB7J5Phnx4MKzUviSo58viEW+Lsqi0BEvo153zHh0Sr/Cu9upKMjGy0A1OyEAfVzXdOhGj1+99Q1cffyS64N11q5N0eBDgseKClDsP+eRSRSxhj/F2+50V6wQclT7eKlfF9tUXqjhv/LmtfgK86S2MvDmEmuzB7dAHlRj7g04/juLhcEiW4++8pLDo3PPC6LSl8CB/SMbBxOILB6xFDThC62CrPa6HsoUXuIeK3E5JuAnbYB5PRFPy6IFFveSCJwMOkHickoHID5xV+A7NInkZtsT4nBgPAee5YzmDR4SHswT/8jNJgM3nHgnUk6iIAyT0I+awGMmbyVnzjBk8ao9tFksqkuCV2jO8sYnolqylVM9zV53sJyLolUmf+IteATmPFhz2IzgL2rFmgtpmLE50Q98UwqyOQoiySu+Ac+xj0FSZYSfAwX3tkbya0vX5tAeAt5tSJ0Vu5VUUy5utbk3Rc9WDdq0csKLEVoIsAP79Aaw897xHDlfAatoLBmfTKkefpRdIsiTULuAqYJ5D7wo2Xf/SjwEwQO5rsTQuepoL2BIfV1dZ2am544kdHECnZwA2yZflRQVLEIgtBbKJBDbUx4z1lRPY9HWI8oI7ymJ67WEHmJWVNvHHPeG0ocCXYzZ4uLxibnR5cRZ+tRE/1fvbahUgG5Z47bNqRwcGW94iO7LoTn3Yx6+CDN2YojluodHkJXFfEdexS3l3jhxUuj7FqLO1D4JZMZW4yZbJrz6luiNiuqj9r5boXQZVaO1chftv/7GPNl52c3Ddwr01TQzK71YEVxS3OMKyi4dCuKGRqYCgCmRcvUFtVZ8VEvkiFsezMYxJ04XiJNv5DSltR5kK9yzwb7gzj7G9FN7Zkf4/g5iWA9OoxhhYkxz8HSIhns3cBiMRdRVs23SjXWxupuPam/iykQjIkv2+mvkpoU4cZRPRw5YGPPLuwuJsgfZjT0sIrza153vgTutHrkCtqhWNTLPFebmqK2LO11tkJtP6XR5K4JgG+xJ5LVZn1q33veupMHm+GIvisVCRBvlkbaohGjNmAbRgX/lwpqfKlJk0KW/oCs0OJRAXJSMo/dcr2xMqtP3ffD3z7D8jP7uORFAO3hJjXXjycPet1EqLtXBeXv3y+uLtx/7TEVDuVQVZ/r2td49qAdOgVIKwJXnhnHNuz3XjFgiSckVne3X85GFju/yERWXzPtQQ3hBkRFYoUZ9xMidexsrnmwAj2AE24qHjgnFfQitHyPxgdumbRGrfDJy70GKWfEoFD+Wq2n/buh/NSCI3G5GwI8TpP6vxeuY3R4MrHnwW2M0MFCfBybKdzfJxA128tewot5lwIHxcuBa+6dPHwYd6itWlR4w7iAUCRWHC8Mk0ISpBIrDAU3eRIqeyO3jO8oi8o/oqRNL9XVdHQz46FfYewSelXbFJDdMHJUu2pvh9SZKJ1AoxJzyy/yODzaaJkTPcHC2xIQZYei7at1aG3kVZ4mJjEsxMMSIPInPz1q1+D8X27DnfXQF7RSdw1a+TBNcn6MooCPf5uLw3qe962mZfb9Vb4qqaNe4j217eT8QxkAbtHD2ROHsyb5Xgdm/3LJDN5fuoF368HSjRVFdReZU4gZvgffz4AjSlO4LTlPcpEpT4bnlHauT/weYmrFzhMYAAA=='}, '11_crossval_benchmark.py': {'sha256': 'e525b1b436ffe44eff8ebb105c902663349c814cf9c874e763a85cf14322bbae', 'payload': 'H4sIALS4K2oC/9V97XLbSJLgfz0FBh0bBt0kLFrt3h7uci48tjzTsXZ3n+2+izsNAwGRRREjEGADoC21Rg+1r7BPtvlR3yiQkqd376ZjxiKAqqysrKyszKysrK9+92zfNs8ui+qZqD5Fu9tuU1dnJ3Ecvyk+icm6LlfjqGvy5fXkc9GK6N35+4/RpzZ6tS+7fSPo8VJUy802b66jdd1EH/Imv8qjV3lT5V2xTE9OPm6KNoL/dRsRbfOiisTNTjTFVlRdlK870dCXdpuXZdQW1VUpJu2uLLpoV5R1l0bfd1ADcejaE7G9FKsVFGqjuloKavH5i+gyL3N4XEVtvW/gdSOWdUPFxgi8ipq6yzvRRuKTaG7N55Nu09T7q02UV9G+agWUhFJdhP1Oo48b2eFL6P9lXUH9RlAP1k39q6j+BXAob6M8KotK5M3JrqkvBYBawSvuDuAtgA5lVIl9A3+WZd62xbqAPueNQMJCzVWK9D45AaDbKMvWe6RslkXFdlc3QKKqQuSLumpPTtS75mqXN61Qz1dL9WuTt5uyuFSP/AdepPuuKNXbv7Z1pX5v826jftet+tVAL+qtemo3du1fi926KAXju6zLUiwJO4Xwq3pfQbf5+w7AQ/Pq20/YGn3obndCV3lXr/al+Aiv9EcYHfX1ZXWre/7X+tLqHvxs6ja3+rIDlsHu7m7xV5S30a7s1Pdqv93d4rtqp17toKPwAsutdHdFflk3Fb5sK00SYK1qhR2n92v1uqub5cZ5SCuqWlXcl/a6BOaoUmaSbFuvRKl69ra+KlqYJe/FVSOAM2qvzlZ0TbHUZEpOIvjvVV2t91j2XQ5fb14XMFvy2zF9y5dLYLTlbdYChwt+pyZHFvq4VMCyLUHjt6v8UyHa7LLelzBN7PLrqf3UFuWm3ouuE/bbrt5l173WRm7XdsVOIEk0Z8hnr1QjYFYtkTaGH97ml6I8r5ZAyWYcfehwCJvVh2VeKq7jcUCmbdNV3uWq5mv4/bbOqd5HUbV1g29a0clqv6y2qij+lm9hMrQgaLai0SPxct/Vb0SOM/WcZVMNIPHtOxzfk5OTdz++Pn/7IZpHd0STGGp32e9fbOMZ/J7kk90zFC2TT9PJ71+8i5lw8ZLFql226vb5pC13zyyRK6vcn3x4+e6nt+fZ+5cfz6Gl59+cnp6evH/5w+sf32Ufzs9fw7tvnp+8/f6H85fvs1fZ/3r59udzRCk5TU+n4+g0hX+m6Sn8Ay9GJ38+f/k6e/XjD2++/xOVIpzu4rIBPKZicjaO4lVT7+p9By+g8v34SIkzt8SZmHxzGEagBMIYAT1XYh2VMHTZJYwXziIQGMkomvzBEh4zgoUiB9BHUZOAOIUZm2Wj9HPRbbIq34oknp5mek7oxQskRjyi+sU6AolLYFJxA/OzTUYMGf8DkQ3r4HsQcbCCnTdN3ST6G02Q+L34ZV80sBhtRAkLXdQum2LX4Qq4LYiPZ9EdAr9Po38TYie/tzAItHZMp8C+VwKWrSaNNWhGrd2JJXTNleopvs2QVbmzZb2k5SKJ+72Mx9Qv3VECCJj9AKtbBGspPqclzRD1+mDX1/ErlBFEMKzm9ll1VBKWB62PP7/nHiACCf5jOizxgbGAbsqB5z9cphEwLyoJHTjljy8/4GzoM4vkIlo5M1hCW+YftZimPwB3tLt8qfkIXjYASRd42VztUW35ib4kK8HdBFrPs2xVL4HPrJppvlphM1QliSeTFcsaGANAI4fpPI9b0paypdSW4qP1J5t6K2wI+PJZk39+pkDBgth+ysvDoK7zK9SzCGJT1zZOOOYH68LM3O27yapobDxUuxl/bg+3DyvxBNFtAQRqAvOi6gywFwfrkjraToDNCMQXQBA3y3K/Eqp6TvrLPM53O1GtrE5dLA6CacUV/mRUCKsgLt88BAj8haV4pemxBva1YJyRgD4A5jLvlptJW/wqgjhMD1beiHw12RSrlahgWLdBCM9ffHschtjVy014TL85PV4dhEUBkirchcPNt9fFbrKqP1c47a0xbWFhFlnX7MVhhoT1fSkmy7LYtV9c2xgnh0BIcSUh2cJIyifUzrMVqKoJCs8ZLWTjCKbWXsxQFyahZSQzrVMABlBJt9cwKxN+aOcfodFxREtYVl/TI6OAayHXq4Hjk/gz4CtQnwLc5/G+W0++i0eow25gQSqtFQBRSwk1QmcsC4yjogLeASZBOC2aL3m7LIr5m7xsheoXqprFFeiEtDhhl2cB2Uu9WxXL7qLtUKWqbhczm253GpnY2JGoKK3BYM3QcMtoKrIgzADPYkUtZp+mUs2iykoYEw6sJcKjVYBNSdLXETxLzKyoQLjFuHRSNfkWa2coRyMBHQYFr2jwld0eSLyMJZ5sUb+wCrFoy6Bb9EkV9V6PbRKQJFtp0KDSdmJF1E3lR/o2snsm5RYBZLklG+p/6VfLlKRy66jXdoV8uyux+Q4paOmqVhkcsPZ4P9kehTbECgpbKq5VhiwrhMWqt/VFml7LjLgWi3g6sVUWBVHGrIoFbZ2YS91Ldl7DZBHNrgEhlXD5mce3xMrwpObpLWlIczOLWllxTMOWXYtbNWlbAdMiB9HRzpN4DNMznsWjUUqTFDUZa0JIiz9tNznI6EQ2M0o34mZVXIm2S0YXs+nzhUT7cl/AFCGys+LaCjTgxcrGvgSRcWEe4Z/FYgF6OHbI+xYsKqcszcYlugNQHwNJU4mbLik60J1UqykPSTKC/5RSalX71+gFqqV5dZtg/ab+3I6i383tIugCwvcgg6I+1IPaa/wKZcTEyAigKenu7KwC/hURihX2gLXUlPQgAXumINB1N1vo4cWCbfCyzNCLlBUrkHBEH6SKKYBgqAMoNW8Qb2DvK5GYTllosysMzMiYvEXAkhcLYAfU8ORPbIt+3+tK2EI2VmjbhAHibx17hgx2wpaQmVuYOWVQtVNFEgv7r6PpKPona0BcawgQoXJjHCPERIDcEygSEsbOQ0WygGxpbqHWL6fJc8E0WKSswSGbjIKl7aGxCl+w4IW38aJfUZQ2QpoOB/HB4TmMDi4Uh7tEw+0C0eULYrg7tGfByPK6oKYEcRbMjHt6gWXHZqZgI4obDOPkLSglHYI3CBTtqmj/WqOco/fUt9GjqvDwhOswqY7VwLUeGTgjF6rA3sNyrcXIyGF9t5NKFrjUhjFlutFCG6AZSBm/0f54BU3iNzAXojszR+7BFm27KNdSw+BaGj4kZOUQ64blOBshFhjpwCxXXbZ7xcXkEiYpIF21iT0vSL6qD3eqrVk0NY3BcyT9+ArL+yOS9pzEpnTPE5Bou0eiQOfzhvADiUvOdxhXAA36c3lLTv7UVZyJaMqUb3CdBG0T8Cfd/dFLGvlAsWYGmrPUtSUMR62ZRWyR2d8sNWkWoa2ipPsSJuVlXZfhFRNVA7lAbvOqWCPdpc9KYxI9i2LdqRQVhliNmVNHO6jIdYReGG5ejwXrJ+v4Z3KgLvPlRqy4GZAbDqh7izElqUlRQW2iTdxWG1SSOlzMe2aD5GBaxnGsES/drYA/jfcW0mbbNUIkuuTIHZmHWjascrZZBYqiQF0Le9+g5z7xBjR6amukUvlQvZwFx8xawfOi3DdCLvoncmbSI862R89Koxeo17+stqTtoP3bLuegqQjQUGAIz06lq4CJE1uk7JpbV0KtpLGFqzpybyL3S9Ir0WXqI1mZcxIs+X5V1DTCIHrdJater2FeYg/RvaaVfvk66QlGBX3c+zJoL6j/tvlN0p9hMF5n48DMG7kAXKzb/BNxwWlPKWHEkdSyC33ZTtQYRxnqrpJuOBeS4KLdo984vLY386Ad5PS/rmrm8OBnxpcdRAk/jMIl1RjMj5J8FNLBUOGmHo1ADXcnVlhxocJArWqX7oBOkn7J6diflhML9ugxqpFqgf5ezFywi1AnmEyAEXIV/MkvW90wdAs3QsJNAaVgvu3FSe9rQFav4zueFvla8CYD8sMTtVg+WYzus/aOuHF2+nx1n37OP8X9mbFOPzegktGMHCv+s6VUYK6wwFJKRLArd8G3ZO0Sr85wNUykYG/r8hNYmAMsZbwUqH/MfH3kQCXpz7BUrgOFma0tT8MDuP3+AVzN8uDreTQN8YqUFvPQ+h5s8hKE8vVJEMrvHgZFLSRq/O6ePgUKgVHH3LXGlQvHB2He29rwUuy66Jz+oIDPW3znwn8gbJiesUAlTfIBwBndW6u4hOI6Adk/6agr+JDpNkllGevaowfvn8Uf6q02uJd6c2nX1J+KFQZ87JuIDHaegGkUe9tvH4SI7mzMnvQxe3Lf21ozvXJUnbGeYu5mk3wp9VDpQRC2EnpYlxh/iaI65DOVeqbxCWvNXNJx7rkcYTENOdxOPEsL+hLNe8CeDvkKT6yFQ1PNs6NQ8Ty8oXguC0d3brV7HvBxRIwb3Tmt3GtnjFEYjBWDZm7PbMKXuP6746tI0attmYnhmjj1wSLV7RsfFJLgboBm90eo8XMlftlT5BAOBm6tSnvjzoCQirsml9WH0CgfH0DVD3IcsyizTG3aJ7ZKeB3tofH4LmIt2UMCEPtm8bpB1nfU+5AvOqjjh0bQYFhUa1QwYCnGX3LspUZ84jio1nXK+gc6s7DndjAGub2ghETUngP8yhXVx4QiS7bvKyP4ojtC7Qmi9mRxD9TyMLqP/vzrWL7lJu8j/pvGnsYsbemy7jK5DZI8XDAdk3OR2cTyhBR5C9vHTzTqBVTbrVKM3nmDj4ZcrrKj1A5ELeEGPfWBV0P0n/LnC1TfFtpjhs3zB09fkXs1WBElkSIYVx8drs+qiuwNuv/zG+LmXdml7f4SR6JNpuPo+Rg/46buPJnCwzfpC8nEbdWml3mDJRMcszkRZRzdzGO5qX07Vxgi+Dm2cHEKw7Gsy7qZx199890/f/fyu/gR4PTmqIQ2taC9/vbFmxdvJDTZVooqXFd0pUjiH5viqsBASBOZqWe6XWvq1PJsXb8G0jgn/znWNTMqv0m7AkQ9bqFsQd2/Kdp5fBOPOQoUDaIztRkNBO+Kq02Xlfltve+S0ZduqdLQgZYG4yXV99WumE+/tRpalnVrolBkSCu5rszGcaICZUSJ20GkkTHfbNYZe3v1m4coGOTw8V1bFCuQIVsddFsZgGA3YYhdk6u9WA32oeThCspyUrXZctK9vUdipNXuVxZQoPJRVOTxaqqkqQsS2jT5ZU4yMyggXA0w201G++hAHHILmCJj3GeoP2c7YMJSMBVQQ2fAnuT3d7XVfzT6XOMCfi/SZb27TfomDU4C+I6zIImdCIS4xDhJ+iUlgVv73nHU2dSw6J7uq7KorhMZumZGleNExadiibKYAy75MYmX+1VOe+T8Gh/Tos3yT6CD55cUt8f75MvdXiK1woAPDYjMvem3CIJhpvx5HknYVN0qe/b8xB5IjPHEobyTs+Y+AhvpjkHd87DdUYtqNB3XmVARnYBPKNAzpUi1HQwdx28nshXccNsD5zdiW6MtAAxqTQE9sSVUChTtgXJGSMF1N+uCbXhlkDIZ9XBO/5rPo1SAFpGM0q5OmCBKezxiUh4PfPRCATXx0+i8wmGP/o2iJaLvcbWvREfTsRHNvkpRbVzeWzZZRKG38JIZDcQzKIMkuITaVKV/LNG0iP7GcYxz+sMLW5c3ZmvzVHG8y+J9p7QzsZ3CNJHVG3cqgxDsihyHV32/sCdkcAa7+CHzmrrLGnU50Ck81TMJtQpUKLaoZ565bttGF2g3+U7AsoyFpkdKTamUhZxf3Mb7D67BZ8bw79FxNeE0MSMMbkWrwB6Oe9/+J9XstUC6gfqIqBKD8Yj2A7V6urD65TAc+vUvJIUu6MuCNUT6bTbwfVKPFr0l5r1o91sSTWb9kqdg7iyi3j/z7Fvp7TAhB3IM9GaBNU8RFQvW2B2fsbX+W7owbTM4y2pkryYn3ogyDNo/Buoo4BeM00zi9rXVlCHFZ1ST6marNxMoUpd951Q+sYCPjTRm4wYXIbRvrCmBIVnsqJAlXcbS7Y29vQEL2nyglbE314gcAYHLy3jW0ZkC0DZ3nRX75cl/jiFr/WghZ5+X2JjKmZglrnbhvl4YOT7nP+PIkvu+scPBuRnHemYtHYoC0G6H7l1DF1vFHaMkzjvoYMeHRdrreIQh4iju3TBxv6MXfr0FibrwJ39ZCu5r0WTm1R/sW9FgzCgF6iWBaBKnr8iriFry9KmN4ih1Svl+VkuN+XHf/bh+B4tvc0syLOh5tcqL7a67zUiP80T/Q+Rh/Keffo621BqwMba0yWH1F6s0+hkq0zyJcHZF03+J9Hoh1QxqtKWDZo3Yt1DJE3ZmjbXG2yWXdTQAhYjzccQryVEpD0JFq2GrghUEqfZOz2R7EwIJbXwCzc4OkUB3oiOKe6cSwtKa5XOG4tFBeuEEilC9sSzhRie5XT3QnFwQBndgGBCrqaB2bUVeJbBWz6ejFNRfeEOn0uBv3uKMxU0qqdKOBtYmM9DAzUWV9IWtJ/Idcpra/4RHTkCnPsXhtYDO3dpDWg4gCi0vr/tdRq0MzPYlkK5CWi43++q6Rc8BWOKn7Bzhd0hzh5qBXSGudWibFzEBIf9rhp1A3UAExsHRG/qtmMVuLrvYL6NpNMedRFQ6E/3q4D60Wv1fbcTyeodRTujfNkstWpgK0MDSH5stmSMTIhR+80NtdTD6DCJTrZYoFhhZXhucZcjWX2cDA/7bDbY/0HbMsrRlCQmm/MVRd+FCLYX15V/F0t43NM47H9yRTYIwSF4wQ1xoOwWePmUKj04ebmXbzPOBNhYPuCWkSV6y/Kenq2UqT+gmepdi0Ci3IkkOLWBSdnNnpDdrm1+LjM4/JzLKWu7ZkidJne10AvnVS8NNFw43JHGLJzpj/4Rnb286sGyac9Zxfxb3D92Gt81fzWVXwvvN1Eb2WaD7cK6P2sUDMR35Deqzzfz5i9PTgdAQ3Hhv5nF5udYKtyNRPAFjHhcnth9fnhlOpN2M02ZmefDGkqcEap94bDv0sb7ML4uy6AoR+Iy7iCogkv2HnruQBn7w1IY6F6x39d2DwgmjPLZRtEc87h1n1oAGDjofhbjNl02dracakOvXmA6DASkHY5ZfiTkDAWb9VQCsVfGpQMaan46C0gxPSJ/1O+Apkf1T1H1GlUiFVjszhv3P13NUHM7G9mAG4ioYOq92ZFYOF+91VJ2TyK9gml3h5jgL3eQhPMYHHVbZEP8yKPc9cWG3h1XU8grZzuuAI/uqqfc73Ku00TFxfuozo9F/r7fXrch+3nzGuD8+ioPbaTLU3ZKwaO2gu1MCkL/05yBeSsV0Xl4gpAWrlXLJHfXAMP6qvtpvc4nMgEBBrdsOl7AtyMXp6OJ0gY6MHkRGXEHkSDx7whtO1VpDsE8WFyGXtbwYu3gfLsOYBNdlFInErdG5TMTxZ5GvkqpK+cy4HA/k0gwswaLLMtzIW4/ZPIVpvGUJp2wI88IRgJE8sK7WPCu6FVCEFSvV4A0lsaG0Et3nukFOAKQ+iF/2aAznpTvP4dNbVJt+qJttojHz5h8WomNOpoSNdr/0n87f/pz0X7/mriSyS4OtGNCaGCPb5aEpC3PiMyzekrDstphJDYNzMPDEtV7M/EBkm1Z89LBVyx05zjM8tZUoyQDPN0GRAR9u+x/wRMdN+PXtQ1Y/y8mg+YMX38aJGneZRMo5OjBr1VFHYK1XeOjNXmhZxNkc7atIvb16XpFRefVfyVWajtel2FKC/4yUQhv+wGMFOvEeaOR9e7SCaYHB/T8JintOvYHJ4WuA1oBSGbSEuXC6hmmmM3YkkhsGjWscZB+CqU2cMVhXxkpARSeVSOJ1knZ42Ma3MbYmy0C5W6thGP1vvxlpsUaSWFSCjgjqXbM/qTfoZggOjszsMLcSoRh0e0dgjT+BtIXpt+xWkOVsFard7Nfr0t+J0hjO9S+7A7/d5qHaVnOkvBx6vaNhS0NH79Hiu7cnVu/AhIbuGxLTm/TlKt/+74RaTSnIAOz2pgVhClN+XsLkYqsgW4llfjvHGGO5C47RvQ0H5IMkpdOH51UHrd++hZ+J4Ul2JJv9UMMbhmMddKUUwtPmt0YfucQwRmDna3tPjF6yv81/S+JIb5RBGUrXcap1G/6u9zumYynA6BTgzHVzpzQA1oJX4nnsdr9FiOmp5d3FcUKl2uBt3uPBefc94sGceaO2Mmj3m3nbdVb1ys3VK4t2+mvI9+ywQUrK/VWTk1aXdXVWAfm8PV7u6lXRGWezbNMv02IJzRMJV9LY9EunmIeM1tJRwN0MDMUJjyjcFZEE7JptFuTTaXo61Me2E7uk3ziN3Nf6IAmig84RjPp7SmIhiLYZ3BRzoYGqyL0Ee+IKI/KnI5j3oB1uEs8bGgKDvKDAqCEbqu3xIu95nzx6AwFnmzeYZnqOekVRtdWTtq3XHfbRwMD1t+/4PXEh0DxQwBSdSLOfjuxzgP39IpqLYEzS3360GlAQhw2j4dR4PnMEerDKMXPdDM3YGm3fR4OnPI+CIo1rrMkQhHHYVnfsdQ/cY2x1z143G2JSwgZP9ko5yweKHVwX40i/1ESAl5wvxfHOG4ltbb3Q8x/MN0/IWUIe//Q/KrFOf/uf1VrQD0LiiDNSuP2ZtiwBuYF4JA6GoWrkPKUpSK1kqIBCbXnw+GT4AIe7+ISPBXGZ3nEOjGCmL3+YG336JHx448SmOxPCInxvNI4ccpX6B2Wz4dmAW/1q00u5+FNbbeHdb4s4BhO7lCXAsIDrpTeDDNPCPFgOp+DU0b1DC98r3Heu6cJTpzC9HxAVcrZcWEw4iaaLC1+0LBw/usnnJcqxVMrHauaNqX/S9qNcHY47wYqfnDnq4NgyKGa+uaTUrr0IuZTMi5mrt1I2i/7qOpKqrGlxyLhAM3bIunjQEuUtTT1lsa8o9s74umvVwXWKiS5j1SyTO0S5Q87n/2IT+vH2srSC4zh+L8B+pKPoy03diopYDIMHqc9fW5lCyNWHSUJk4LfUjSnWO8V8rv+wdvVhi/oI2z7eKD5uDivv5SFr+P+VZfsAy/u/0eblwfnHM3l1rAbblWQZKNtydNSw/MewCD1rz7UM/3+y+uz1V+odahnWu73tNR4gbSiPVKKTID3wGOPg5s2BEwe9fURKl0IwQHuvErULcHEkF87CZRuCYrLiUN5vWze9tw9H6VzJf8feql73eu5pPPJine6wcv/5pzs5UzNyk5e8ObxF2dvEQ1GnXPYj5xSSfwZJnz76bhz9sywaTj+dyPTRMIb0LLdu5qqdlM4WWUd05vmNkafLbb6bx3+kiEbz9gZP8bSZObXzwqqBB48u84ZTDNoiNr+xjhDRv4fP+jz24A6xAoexgAm9zZtbStQxdPzNO8szfBwueKyNEp4Z/xyeiKS8Qhx4PLe22ah6Snnk5HlMN0qHMu3xqZBbDIvwbIJx5FunbmhOuwR7F2NfL/iMD//LxOc/i1DAa0q7dJe3id8+UCZvOT5ZZYnUddZF06rhkSoJE9rpcIAgujXZiAUzv7pKeu4a1fsMd0/nSY8i+Dr2vBBOvbZbBarB21At7e2wWtP0DrRmYd8IZGmilkMX5E7KqSnMVP09nhP8bmQK0OF+mCqJtStJ5LzQY9H/0iOPXQoqNfNwUez7AnAqyypPTq3eYPZpDG+prQm+zHeE89n4xIk4mMeWok2gozcqeafplytRHtgnZxDsUnE7iQ+hYUbKQ+OGyibxn51AVov/sNCtLPQOWo3IPRXllKCU8wtyFkxbTqHwcqLxZwyTr+Hg+ycoiEI4rRRbTLcC6/xz87YUV+irMi+ummKV5OVuk4Oi9vy/QDqSXd7t0fnx3yIbQ+KB5aERPyXJPJVZfjHyXXRmslEoxcHJR693xSfgP5ZgGi6diN1vq3bODbIonrttHZvB36Yv/LO+QFGwz3eJ7OmYbwWR5y3X224ep2drbJ0W0v9T/qn64x6bB+tmfop/YcGdHuEv4XivQjOwz/Rv8ZYG9HFRktc+v8togvi3ZTJUSvfoMGMlevDMLNpzs7BWOTaZO2cPS2l6LAWHVKMwx4h/4JYz09pvB7McW1G6c8LfPUNmKZ76uwyNXVi6tf4m41wXVlAVqg32DRpgsoKFL1XHkR19hR5jLmO5rexylkUpE7yq8vJtZmdlczLBKjDcIc7PW9drDitf78sySexaXqAajWReqaAf3wnBt+38dvBY22OBlqlEb0xTS9R5X0gtAWbYl531VrmLWEayU7NwEsm52WHH9Ns9mkBcO+ofX/5L5czntZeGEjdpZXA5Q7CPMpNJRye5HNPOugCqHXlbs3JvBf5Q6khX/FpsRIFlJjmop1l6hSgb6OEinBp00Qs20kohs5O9ue2c9zB+BuscoD6yN+ulipNRwFjNSxs9C4ZdCiKjHw89CpeluWfwuJAnUnySKZL3oVibhgxQ2n7DQJnEfUihPUUnT3c8sxkzXI4VH6msDJR5Bd8Phlb/FruFj9w1/K13DweitsOJynry5WDmYLXPOJyH0Nt8PFxwyORT/00wZR7eEpaW9dX0VPPzAL0nAyMfTDFoT9eB/U7+iuc8qut4MZCHzZn1Os8ZVZkRtPsTP/ew17Q6xugeAJZnsh1EpH1hHceS8mFu75Sp0q/skjSJeXAo+ted5dHfnBlqBBcmTskeIlmskgNyxcFgMdai1X1vxezwyizjGmz4Uszw9tsBCcbCugcR1mZPnlOaTd2aX54Ej7oObe4ecnDSmN+6rGeB9GIpBst5Ea+W1zxgmvN9dHwnhrtyGFqN+0dj22AeE+yknTdaclHKjcTDKx5p7IH1jq8XUJoEP7m6hH2/wShwhjewAWtZBvNe+G34SN3QstarojSLh8MKKQw6fLf/OjicWjRgDjGpmOl9k8ApQCQjWZSL4a/a0jzUDO9wHCqh4hf6Zex73r627xN4isc6v3bH/sDZxKFl/yFL/rHlnm91O0ww67a3B9Du6VPDfuMD4SO+Xj64og6upg9aSR+0ik6o1IFO+UsslUdqjWAczw6eLFWLKEmAgSUUvw0voLYAGVD+rIV0QMkiWQFFWGaEy7AgwSyjVtRFr5QUM1BMCZxwOeYVzTTDyp9klngWDXFOn3vCGYjN0cqgMqGGoKdKaJtPxWFpnYJHxkJycWEH8yz8pX1DwQv8W2/i9wMyhoWmt/4HzaxDZQbF53HRaXVYjt0iJBKCxQbmjktY95srHqenp6eOlAyt5dJtIJWdoeCiwIgE3od4/JiSdACnoLpk4euX/1J1yQJ5UF2yyj1KXRpwfwSTJ3zhkmQEiMVK6mVo/dGSJMB6Q0vMgxXAEB88RgMkOslAnIAa6AYDoBP1KA1NqCF0OSzwh8M+g8UDIZ9mchyP+yTKBsZFn2PDkGbfUzOZLoJjr2SPulLsqDYXWxyL+QAO6YeWqvIYMWU0oQfKwAepVs7SNrCaBKo1GOGW1ZWK2axWmQOpn9vo/su5V20pMCd4SZGca3CUp/S3lARa2BwjpS540DMVu8Lr0QPqVv87WInq92N+jykTQ5DcYNws7wj2AahhVSsUCNwrOYiE5/Eb7NAxXdtAs3x+B6EdUu/VzYHyirWALyJQx2F6VTWwLh9fW1DhtHBnJXsxtqSriUj0E308PGb0YMIPfSM2xllVeZUY/80oxWv5KB24/mrUFfn1kffvqVB5GWAJpgyYMyaaiTIAY6SbToSn/TbkATfIhQ7OaKWFyhpUQ2XVlZGYatvoT1JFHtvNjq1W3W0lu9EhQBZOYwulACDFihTaRoqXvIxLBX2poDdK9x9MuuD637T3b2zfDeZgbTfmvFJtZgfbsvsz3JbdsZCu6rTo98NgOKiv9gl3Mqit9nsfwihEivDHg4j1iXwMMbsrF6S5qA1eua2qeaDfkWPl3e1JWirVhVYX/lLqhFXayzd7rU0MIPvQVcIJD3o414Qbg+5HqHvR6JZJ5TS6UPeHYba8xj0t4xSkvIpd4rxz7sotyk29Fx1dYcv+ffNKbhUxsprFndwyq/wT2DfZJabGpds7GYb7+jAcuXcRCkG1zPZx32dvv/Q35s0XLwoE45EmtHUM6sekXvNvuVHCAUdWXRPv0FMG9YXwSpQq1NNddeVFUT2wZ7aw/K16plZrsvcf2zV3qT/SQzdm9GRw82/spQuFJh9LcQIJayXOEUyJFEbIDtM6GXKcfjE2bHndAhYmCL2HB+V/JWS61k4FNjivvRC9dnC087JUF8awGLpzAN0/dqgRvx76tvEv+XB8yEAinDPFsdKel6K3nynLWKV0fUUvtsa7RlvJYGUWyUcniZ5hNWUOhcc6ttgAXbGDTBEPeHTQzRv+0ruXG90QfHuUXGiNpo1vVfr6/le+/ca5Dd2ajj3vht+GpZmFrvbol/QuRLdblguNVUe+cXNhYWghhsgm6sgchZpxKhbrrBqfbnPyqFHUGkczqQRIDEPFLHHmIeeiVI6awLIypnE0dl4iIqtVvQalREXW0f1zwDm7upFBEvY5hOMBcI+80EqyqJ8KuXfiAQeclBFD8a+iNyjP6a7dd+fvP0af2ugVAIOZTI/v8aajP8Kc2WDksTXbY/v3V19F/xOISPbgQJHXtQq3tcEDiTAutOKjCW3EKonAI8MdahzdJq+kFVMyfrF1m71fuY7yqN2iyHqVAwN3xZLuOZmoPIZLto5w+S3F9n8c6M070W3q1UCBdTyRF1Hpm8DuI74QDFPx34Vui7oH++7A3S0h4IFbruQFLvXaLaJv101P1/eRurWJ0xKDAWpaDLbjY8oLujouMovOeLtjHE3tPWvs6ZRvGmYFP9AXaIIvLO73HXd49lUrgB8IBsZZOtcVu2Bo6LFJl3m2OSrOxFZuec6uNeFoj/azEDteyqZnkVTJCRjq3XjIbgz/TrGD1KtTrwtOQK8EJh1LWO/ZaXpGFUs7ijeaisnZszMx+WaIzT6oI6/EOrMojr6OMLg6xfu7zT1hw0z6I4ZSQad+ZD2M5vB7uXgN1PpbRPdlRH+LXuncnvQAZrXyB6lnFa+snrt6NzmDh4803FZpfmEV5xeyvNP8ZDJR/58N/yOrmGSARmeSso6CD6WrUzrDZ07IBpUlPYGudPFSosoVE4j91tbI/Yyo3koIxRUvyLWMtV37sInrD5LuCsaU7p+5cJfPhZ+fhSxlp7y7ii564TbtYMrsNQy3fREBPBFF6BeicfFEjeKTxSw9W+OHOABGFlYuPlNYfnDzcB6BxVZ0oGX1pd/MITgDbfcuZzPkkqlvLmIcUJhGNG/AQKb9STpCpHaMv4z1TNbc7CFcyK7NsVbjDjOhLu0aXMMsyOqNs+jbURcussBuxivsnPlkjnQVZJcX3SfUlFBJoeTqrro2ehQLTxwWlgwso7KXjti6w3Z4+P/j3/EakBU9PIITKD0kqE6r6M+OwRX9eNmK5hOrGX8fg7Cx0Luz76QXDXPIVXRkNOyYyl5LLvBXDwE8cq3aw8h7Dag9HpW7J7x744VRf0lXecsEWYDVe1LV3bOb4bCjQ5s3i2Beni+ZE256qoNoGqzcXZrjbYeaczj9CD3WMAm+sudbIH92HHiHs1RrM0q/kufy+GjQHW1c28w/uj8KiJfmVzoxCgNZPhiAexIrcdQzk7pBwbV4exD0uS10Sa4x3xkhlHdeKpMZrWuSNVkgPRh24NwkgFMMNATMH6GFv1m8xVgrpWW0FzE+Z79/saVNV16E+nqHvTKwU5g1cRuQfPVl8MKM6jIpCun3gi+l+7gRUtv1+uv3fx3bRsO2qPYt2xOE0YQuS5HMJg0GPZpgpPj3Yj99eqd6PiFKzr7GcXj6ND2CRvxxA+CW/ladWjNaWEgawdfSfa6bbnPLxi8mxwHMSlTU8VKDUngoxZR8AQzmZb5vBVk5z1/YRiZYg0Dzy32HV/2RtQWv0+h7ahQeoIIym33I3X6F1yeS0UnmJKXqwUOkxpDji3HKHDm+ttZJWP1rh8QecBVWhfURLNlyTI0HkFK4HudomWO0DOcc0hnDrFOARNy2a2qgoEVZDyfbP59GL0GWNVcSnj1mV/kOaSfwKnUwVdEer8EWg/aRtGPaevUg67LQNGMpvSGYluQSt2eLLV7OjncS/VEOJZZSn5lOFRCy8UHvd3hnOm4JIoPhBclNgV4BYAUYgfxTjXuZS8zJg/liNsUSF5IrIXEFvsAb3pZ0EtYHna/yHWFbNFHA20LOlXYnluhbiXabJm/xXuQjg4eKVrEtFBxUt36AwQQVXOyO1Z2QuyrkQiH6MDd3dR2txWe+2Da6bPAWRbxapeRdhT5MmJlNd6sdJ1fQRBvBCKz3Jc0exA7phBTKdf45bnaFvF3vkOAByD/x5gH6s7Fa3RYUNwsoVkAxvrhRTSRsvGCabPNb5d3A3RC8mbsN4i0oexpOG+BbvMQKaIisia6w1Yq5HuN6ULsGDMtb2Z5YqSOul3ipW28i4H1i211HFwfKm4o0o65hkkzYi23jtLD3RTAIm52gSLwk/ksl3Rok5Udycw4TYcT7bj3BC5TlwXdoJb8SSqdJ5E1r+swvX4SCzlTpRl1uYLTU3bamdPQs4nXNXpNeOCFSvxa7uAfk4BUyBfLQSjjGlNvk+/Offnz/Md3arkO3hNwlGfyuFsjB77Qt4nhIKMcd9AZvdkz/b7F7A38Tu1NgsXzGk3aqyPc/Za/P37x9+fH8NV3EKcu6Zqy6s0d2eubHy+N3DIZBkKEL4yRQZgO6lHzMdWixQNS62hpeTIWW4QS58bPWitJqDa9JDjS2prs3u411YQS3dVXWl0n8NB6Nwmfe8BZTHPYDHQl3iOlKdR/aIbmhYI+MzgSFmbhcZz36ZvE0Ku4PZPig8kpZOe+s6OxgTrze935qvF4RzJNAmX82MGmStrstxTz+vIE+Y8IL6/6skPMY77J7cThY6aO3SsHTSrTFVQUyiZJjSyewyejxTN6SEY+cpj3nd/Sv0dnp4abxsj//MnSVgJCErUHLagyl/gPiv45dRBi/VBf34m2E0OtG/LIvMKWzutc3j36antItch+/QR0U7AjOTG8u+I19Xd4SDnOSiwnRxmZDkDZ4J5JkH3l6bS5/gJ6LSw7Vsu8YL1Z8UPFKNHwunovbRZq67noylz7F8EuBOVF+0FYiaVV+xpuJbezmbAgWtNI3+DkiekgoGSutHRKowYJK2HrpJfrlWOhqnw+8FkvKEY9uPQuVsd2cvYdtMYqu/JBb352ry4Bx0Jw0vAhaXMaJ9ehiyORUJyPQV+UhEe2B1NedFStV9q9tXWW0+W7TCmmup0PGw59iUVhJ7p4+leeFkNgadjzTw37vyYlrYv0Mt3loSGdOVknFS4aB/fI+G5OkqvfNUhh/FEVByIoozOWXxACxE3/rGjLgM5M1ZfSdV8k9ykQfNzUl0zE467chZPXHh4y6l3FTYbotGnyT4iU4oJIVvwrTMCbmNI3MQddPDD5OLCtpsSRAr4tdtqo/V5jh0V33JMRUfU16OKXSHrL7GKKqRU5C0UZ4eCjvYr7IGIeRQhVExYmKKMFXl1PXeh/v3eg7jcqmrmEN1fea+fGVbiiGu7HrvfeWO++ruCFVyfo0MvlxMEXmvgD9k57M/p41Xf0twdh4mQmk0tmLSvex72FWMz+K7mg7Fya8DlEEEkkYjpiwNoU/0EBg4GAFFZ1huXhijcmTxX1Ej2PLdFjHfg13rLCSejNmtHjceI9EGyWxHRs0JJy0J1lqB1I0uXQdrK0Dkmg0VGVO6CK9ZPKuSsoIInC6Yv5finOxeEc1Z2VjU2udxxyeuhL+am32ewXoynVu3WYtNQslXgppQweecceoQXuSsq4ihnsVPUWXyXmbGOCmQTf8K5ZF4bFl75MK6JPkdTI03d2HtlPkdcfI7u9+fH3+9kOf2a0AN11xYe4wpxEzZZLA5dpOIB7tcXCr3mkd1c3wSUR3uPWYWTcJ9z/ygBrkemd6TNKjMHkoZAKVVnOoq0cf5Zp1iRPI+3WEKDRjvT2achU4vOof4jSa0cDpp2A2iAE+daMAlR0v2bXfVUnH4zIgDMlJitfLSvX37QyT3miFIbiBBp4RaqMwuFN694AT5f1xdYMk+arXmYVnuPTTp3fDmX9YmNzR9pE8/ThcWEVq6BOPcmPLYNCLmBg8/DhwAPI4wrwz8Vti7O11fBnKao/PD4pcHDt2b9pxEz1KeE746gJTl2fL9pMnBg7MO7t+CjXjcWQloD15IBZ2ZOwXIGFV/3IcvG1uRAJFwMOxcPyJLD3cxOBNAYv6PGanq38dcUE+2flzXwSaPJwO1rYYcAoq4g2JN1WsRyd5RYYdrDrozmTpbqkVjpSUPi3yVw05cB1jNv5L9UYfJLC8Qfr+9tTRUK2eojpRXSV2N1yT9z31BLRXpy9PdF+e+DbyexU+eye7QbeAnxR4nSmqAlmGN8vGWYY+oiyLVfpwyp7/n/0HOlP7uQAA'}, '12_overfitting_study.py': {'sha256': '7cb8e8b92df0afd9062eca6372485598c76a6155ab262a2cfa1be8a7cc81b46e', 'payload': 'H4sIALS4K2oC/+097ZLbNpL/9RQ8pq5MeSl65ImTXd1pqybjcTZ1duKynb2q1ap4HAmSmKFILknZo0zm3a+7ARAfBCXNxNnLVW1q1yMCDaDRaDS6Gw3gi397tqurZ9dp/ozlH71y32yK/Hzg+/77Zrfce0m+9Cq23C2Yt8iSuk5XKau84iOrVmnTpPnaK3JvVRU/s9x7c/Xug8e212y5hIw6Ggw+bNLaWxVZVnwa7UrvhrGy9poN8+pky7xV+pF5VbJO6tA7PxvVbFFAa4vNLr+pqWECaKpkcTP6lNZsADUtIYdXsU3S3Ltm+WKzTaqbyPuu8RbFtkwqxvNZUmWI6/MXX412edp4OdtVSeZtWLL0PqXNZlBvkywDCGxqW1SAVVMV+TrbQ4/Xuyyp0p/ZkuABQQQqs6IRlbe9BBRWUHYgqbNImhRIsquRNm8vL0KvGb3//orK//jm4m2EtB0MgGRbL45Xu2ZXsTj20m1ZVA1A5UVDNdSDgUyr1tAr6L74Xi/kL/4nS6+jXZNmMvWnusjl720C/RS/K0Ch2Mqvn9NylWaMI7KAIWILalZiclns8oZVPL+EaqAZmfcWa6WMZl9iP0X6Rb5vsQbwqqgTDRGkHuJa7vEXDmOZNTI/323LPablpUwqAd+EhrtcyrSaJddFlWNiDSQSqU1RLTbGR5QTTJ5zNOubDNghj5YMWaSoUxoj2ZvLCxMqS3P4G2+LJcsk0OtindZNunjH1sBgNRQ3y2yTPEX2lPAfYMwtCNZU6aKlbzDw4L/LIl/tsLY3CeTevkzrMkv2IeUliwUw7GIf1wtgMJ52nWRJvmDL2JW5kJXFW6qNpy6Tjymr4+tilwHD6vCrsf5Vp9mm2LGmYXpqU5TxTae1odm1Mi0ZEq2lqPi2oCpWVsUCqadY5nVyzbKrfAG0rkLvfYNjXi3fL5JM8h4fT2TwOlomTSJLvoTfr4uEyn1geV1UmFKzZjAYvLv4/uUPb+L3V1cvvan35fPB6+++v7p4F1/Gf714/ePVe0gMzqKzs3HowR/6F/4ZR2fDwburb398ffHuu79dfPjuh+/jyx++f/Xdt1SA6HFH/+J/fg4yzJ94Pox6DEImXp6N46was3M/VECbdLlkebxMtwAKUFrWsirKYtdAOrau0j+xdL1pYuDVZE+ZZ3puhgSL621RNBugIwfQ8ytIGrPRuZaWFyA947pZ6tD34fEOnT+6Q+e/zw7hCJ2zL//vR+icjb78XCP0uA6d/546REtx/FV/P7768v/DzOHdkOg9uC8vfld9IWVIIPB5u/JP7ondxD+3M+PP3JniE6tg0v9f9+LXzfdWf46p4O+7My/a3gxBt1iylZeB3hEvQL+tPyYZKoq7jAXDCYGhrgzqAurIASj4oGPH8TBCgyPG3gf+eKyKKvul3PtDKp+uPLABqJqI3YLSWcua8b8qAdS8d6Cbp1t2VVVFFaz8d+wfu7QiWyUrwaABo2ubkpo18e6wontRd12yBeBm2g0Rpsaoa3Fss4LbMIHfRdMPCbEWU6oQmvu+AO2vqOg7ykgrk8lHcL9EvZR6jMVEDyy0OYG7iPN0jjq2HOA/qqcCEaAi9E8MEv/DYSoGxlcuaoeRvfwrNOEe2sE3F++vIPfyrxH+EmxAZlkM9hkMkTf6c2upRd/DQNdlsmCSJSCxguItwEW13m1Z3rylnGDJ6kWVlkj1aRwviwWwjFYySpZLbIaKBP5otOSaLowGoJHssmbq1wla0vECNG0YvYV/sPxNsl5njKoZVTAzoKJK8ND0Q7VjBwtfQ9MjmIflrhkt00pH4hmv+NmnoroB7nu2ZVUTE14v0Dg6jNRDqxROiMOVgl05IhcD1AnGKpumeaNqf3GwLHke6hHwI1XxiBrY7SLbLZksnpCJPfWTsmT5UuvlbH6wmpqt8SdHhbBy4vLlKZUIN0tLjxUwvFbH+RkaQQdHv1lsRnX6M3PiMD5Mj7JYbNxDcX64WRAHKQgh9ohBWIEByUaLLC1rbQxqsCtZ3ACz+yeUVv6eQ1UIgSJq0qWDEBjomomXu20ZoHib0CIReiBodmyC3hOSIkpo0hqAHq28ibY3MDEC/lHTJA09Wh7i4kabs7jO8HIF8FjgfwJ8GZrXgPvU3zWr0R/9IbpGNmBpZ5pwRtQiQo3QCQVA6KU5LMfN9DnWU6OzKqkXaTp9lWQ1k/3aJjcsBuP+mgWLWHSHGIs6JN0BE51IMjFoMZi1v/C/wK/RCeDbToFgOAxNQOOL1m/lqfTDTm7Xk9OtgVwzU9GV0JlNbcRc9Zj60i/ju4G3yW2cNqyaPn9xduYGqYsM5NnUz65XyGQdGKvX2uecfsmhKEGGp8SgsfA48e6RHlRPvLyMkJqV9DLhqCXXaZbC/HJkg/yMqauYiZNuQEOKTczAQgj5MM8Fv7Zt17DYGTVHMA+ACkECLCuFhOAETT2UfiZfsE9gOp4C3olQb0hnB7/jHmsrMojX40VzVj9wUNzfJqAhxKuxu3rpV3PVB9IDhjlZsymvBPj7ZwZ1LdOPKfLi9MzdIvrhzo90y+Wr6/K2QKqTbgxXN/tmuk3z4DzUOWIY9tQ+BTYCfSRfs6AfvNPRe8HCsOBXbJ00KFtxBQ5OYdQ+/uZVmOnEws2uzNhMS9ZABD9T2dhoGPh6NtcyebMqFZYLngOTBeZ01bBlAJpawNEYapr8NqlvoBzP8KbiV5vtaDviekNgJM6wnnm0ZUnOJ9fZcGhVwnGUpYVjXzAnLz4EVRpWkkWx3YI4HA9nZ3P4nzFLgTh1A7UFDsSGRLukJtoFeqMolWj0vXdqS+UvLFkGeR694bo4JwkOfBynedrEsWLammUrxTVpDgoiWoZCFMl0ZTJaGV3pJXOELSnmEU/XBqfelbjURC1GQwOlKGcNKqQwftCP96A6wxKZJpk52yDrdbJn1fdFtQ1a3K1ZgEC036AgQq1DXehvr17/GHSTX/IOBaJjva2oqvumMoyZHBDg5k+w8gY0DFxPAUJyjzz3ufOppCVo5h5nHJ1eXLeo5VoFnJLmYLLkyxRMESGreOKtcyZDxr6bgYbarTt5f8qKJvdOUjCX1cIG6tic53GtVQOWyqiWVDO21KvTtAND+Hi/cFt5Sn8MUWTNEFvxCb0M9JaZvfLOw85iLKQX3++LELMA/+EsDKi4M/ggbpN8B3Sz8sDS59mL3TKJ0jpOPiYpzHDl9VA1EIhWTZxkmaiKU4p6AwSw9TptkAlmCTAcOAIzD9eCvF7hVBIcMgSJg7ZAAF2ijp8/H7YDb9egShO39JYVljUUNPaUAquT5HKgPctAx1ibRT1we61h4JavvpT6C29euE6m2u6WalrgptogWyxGW4wW5/FXwCOg9As4XTOqN7vVKmPcbmhT1ywHXQQwnXJsv5XfIPk6jKCjuWQf0wUyMS/HPwMfh94/yCwerAkMFPRyp1w7LIOK7OVBX8JwsKN6k5RsNp7rq0ET8Gk7052G86FT/qtErjnJktKpKIsNo6YIeIc4hkXZpFvAq2q7SynRxTLZ/rfCkzqCxloCajeral1EZ9XUbDSrDDR1B6YFafg258YgLCo0KXAjm1ahS/RZXeUN9Gf/Gn4GVvVTW1m0+NMQWY7p0SFMOw4rU9yh4w/9eSjgDEAaeyNVp5Hpp7UJZnlxhW6i5jtfWNoh0vqlpEGnB5jVgGHCmt6Sjvmq1zLg6gd6AvZKCbxmNfqo8hsh5lUiqFAN66TSAgOpZ1xANgk5O/lX64uGFJMmykstqIG6J68KdE+ufY9DsXp5f/DGuvJJ3EpzKxhacy0D5onr3RZRiM6sTNPEE/3VylrqsESLS6rbUPzYI4Zc1k0sy8yCm8okjept7gGObIkz6ZgookJVNdCGjz0ujNCH9IYFImvoPVVVGTW1YiEi+21dJaTox00R5zC4mkum5fBinRKjEfHbFiyYGiHaqR3wQm2Pu9DRNWjbpKQNHfMb5AKPnED3FyEJ2FXbOHCIKx7/4O5j3bDSbsBklj9I9iSkAP1tMETq4YLkRL7DTxG7bdBA4V2WDoPxEJYXMDw2sCbBsgH/8sk5dGLDLR1RkeSSvvLWbGAw37U+kieNUzHNV6xChY9ignSVR0oRa2yVRBo6QGmIxVxuB1rVEupyaShJ2anHnImquEY4o79KLy8+QYE7o0KfpIQ/4dLCtB18NdIAYA37M0PjsArK7kIx+dNZ9WG3RjcWKtBHO+xy0nBwwHNGWB1zNJHoD206D11VHfYFGf4gp+OPN9WbpfuOnEC2P8kJZPmYHO7FPpLda1Y2LXPSjwBspAqtkxJ4CpJm9pDOvRFPN8g+1zYFaZk0adMWaIk7D90AqkYTYORmuBEgaqVYHK/6BKuIWsi1PU76/rPKs1Ywbe3HP91MudrT3262VBHuOmOE28cTboXbMm2RAXKWgJZrLxYTxjsuu1zYUSsxclbARUxtlb43vkydRCpzEwcMrANjezHmOX+eKsO5uyhXLLkZDHS6c0KcsovsX6rQZGI/DDBcpnxbGSM+M1jjIt3ioK3hWCOCalGH0pYEBAACiEkw00Zy5I3nuqfMsgk0I4zb8faEUgmWTFZtgGxRH4ekM0LN9KR5v0jmsG3C/Jho1uvum3a2cFVtHCyhyVBVom/ut7sEMUzmXrxgWA43fh8a+yYVQyeD5YuSHq9Tnc2fwbf02zuSfrfuoePeocN+IaFxfT7XznGnjnSt/8un8xCfDh+of7l0/uXS6XPpPNDrESuPB5k+0usxPOry+JdzosfxYDopfk8OCBW212pooVgQxFqOR7bMHcpACaaJLZdCbeGZ2JsuvWoArbcqYWLK0JzdNo4uDoVYdS5mJy9kJ3kmBJU4WF2sGpwYBnH5CPctehr7WlYvSOSpFkFruxn4ECw2RQGMmeH+ZwwUjcWhqgXHQcVTdZUoCldCJeiUrXw+h7MlJFvbY9Yn/DOH/wxFyQJBrWzeKUipMiBgh6ZFDNan5uOsWXt2zggDQLRijJ4CicGPhuXQt90Wl2MWENJWOEDNI0zpZwxVxAhEkKHovJoewh4xZC+2ShRX4lA7jsmX27O55b7CQiLSCYtZJ6UmzjgRZPFuzNfQDYtamobHjDAMeYeF3u4j3cW4z6wMV600bXjlwj/S3wCg5qzF5QZrTQyKVp3oQ+iGo6YAkDfphrmE/IOhZG6PlBPU4QuzyMa7y2OOhs46hgewOO7MOs2p1eoHTuQOFkHED0Oc5PE62fPl9oAdINZ9J0UTC07v2EFf18k+r5N9X61rK7muAzwBHGXFenzWTtMeBhj1MHG3H8JD0+MUm/n44c8nzlaE3Lp7+hT6EXocdkLlFWFRva+atpGO6okZUVmUAS9uBOcIWSwHAkENvUEbrFCDb90AwH81y/EQ9kexfgVakYlzeaA1BYwX3uUVLvgooJYRmnyv8FOvQzmU4kVcMi7meVMGd1A9EQaRic0Ek3FmXEqFUgqF1hQOvYNcktQLoA/q5TMeTcz/pbhe8WfucotG66rYldf7wG4fhE5Sc2kpo4PbMqu0qptA1/Dr3Xab0KZpYAxrhyRte6IZrdZkvbZUGtH3adAhBQbI+RbnS9II8JZSDnCtVeSPhvdTU3kPjZOOh96IpFvfuOhjwcenMyIGa6P5LOgapVmxwDA+2YgWc7W4ibF7sVoyH+ftelRopSPMEpE5GljJU9PlbxFrSQhIgcGp8JuHVRp9kpVQwtDtTG4DMAlb3SPiDr0MvSWdlCiufwIRdwg+XbqB25DyrGgUs8SLbFeTOaNsqviG7SeoYoefjZE4/B5NstYZukrXO5h9sJxX4uzEoWh0cU/AbpuUA42FOHKhZ9IqluzhnhqqV5Z7Q/vmvdDdOa2FZzszuwFtLWJDGez8kzIqtAj5t5cXsFrCv0Ee010WOZ0HeW5X2XFd+nQNCZTFmymsOFWjJlMagSyGuXybNvvpCzMHA2OnfrlILAWMLnugpQ7MnKmf7BpbReMOZr7bMtWuatBtymO9watUoDM4uBH+Pr1HeZyjU+u6qGq7S9sU/Y11Mz2Lnn9WlO8l+8ISeUthSmXWRPXumm6UQX/VeYjZ5PYNxl+H3ouh8m4lYEAGfBevLGAy1EMUeD+nZYCVhTqzyK08TfY5tRGjc3f+LZCSVz2bhN4ZrOT+3kgaz0lVWyc8AqCdOPeOBbIGNKD3TYOsU1gmBPqxp4SSSeDbKSBhJu2n/t5K2uyAo/hhOnPrcfpHayiTrNwkMI6d9Ntpcmsz7Bpk7zQnMk0Fa7lW/uQ2Iida2mSMxqOTddukeF5hNu/k7Ls56AjENv9Ntkl3BAH8GuA5UsFQV38nVk8MQFBLtsVHuQeMjDEbjeeRyL6+Lm7R+5fki01RAYtFZ89hVIehB2rC1N+VeGw4Yyt5mhJ4Ediz5F1VrOTfteL+XjDCCOVl9z4k7baoUPgk7ujPva8JSJwFDXnDIQ+j14f6EWpN3HvPzMYdy1FU5mu/rbUGIxHK0zk7WNzKdDr++iz0iA4pEIHVU59a9hUmi6yoWdB3NokaB/ZvkTAOmTvdAD5uVvBlATNZbkp6DVBdvtNa3vZ9PEKimAuXcfrJvOnHbcK7bgNy19x3Ooe0AbnLHoMy+5EdUAVQjef73HRqx+lx07xo7EToPk1ATZNeYQt8/1wXtiB5/6QJWzDQpNMuQD8ydDL0WpyGphNv4D6EGPhX9uVjWBWo/AY57IOM/ntBg879Y37ooE/HOtGFPvaEweCS11GR1PT8uVYGXgq33yg4wbc8Z8QcODfJ+PPuBPgT/HoyxyhTMcGVmAJJpBH1bB51V4UWm5kIRHN4NER+f2yDLva/ftFzNG0KYoR+3PNIEb/vTNpvgXtfrAVhl+Zg/e4zWOBGI//X9QzdbA/p1/izjMkhd1Qf9r8tXsecZI+lub66dzhEqQjmZPPvhDi5p22zCZ74TZecC2GFSuoNTHbAGMNPQJXzB8dbG5/UmqTAQ1vkqifKD2w3WmVJM+koPMRxIO9oJAytZ12ly0BS8Lmmlpy1ZNqL0rTPPWJ8o5uo42vw4wfCn7WEkfAXkgfMWk+A0utKt8EZ7VE+76nGCcB7K9SwVQH6NC47oOHniyKbPu/08wikoZhZ+tjCEQWX0FUm/CZN2nLyD2temuqkFpMDepi2UtlKASlk2olLoYNZ6p9UujTlor3HsF+tUF3VEh95xL0N2OrsKx5VMvg9i7hvbl29GPR6CQ6dird9C9oxalQe5VFNI2RE6Dm2ltPqN38Mva+FeuO+bDIQl0V6S/4t9L+pbM0SwZYBtQADfOp/g/qsNtbcDoorcZfp9FwToMDMRXWdVNxHrHfFsK8s3r5TI34fEjuPSAFpcItAi5H+zOytN6v48jGcXe1yfqNoXOPFuv2sjRayHiuouUsfs91NhK1wn717O9GpXjVUvCfWvj0wHGI687WrUebaNGzz+aeveXFVnrCN5tq+P26B6JeCcmdZoBslYlMRQ6g5jBY6ocNpoWJ4sRPMIAkvUmNxBoRHKfAdGmGtxsau/pFoBpMyijEEom4XYTuURhBeOwf5ZpBzO0pHV8SNcRtY3vhBwuCAz9ZgvdDRhZmBx/w0n2cH/9A1xVrXslTAgIxFscKYmTJa7bIsIFknxtE6o06TIcmlt9oOjEHryqiLh0upVmQF0sZXa9cDijmDQGi/ykwybD07XIR18h4ZN1JWuOuz8v+eG/Ky7Ru/jpoba6pmMtSevdA0tofGn7TzA6eWNWNmqqG5xjqcpWmTSoJq21Pt+iBiUWTgnwZ76c+tw4oASqEoesSI94sR+jEYPCRy5VDEimxRi1Vpk1QNim3aSBZYo/z53A5Z4RFqpt7eHybD67AvaVC7pjJw3Y4JwpHozaTYHwqslAzHv0yWc1+JbIUR8bMO8oxDe7ZB7vc7r3o4recyQGhwJLjkAODh2KCjNfeBOWORlZ6DlHQYlbAYRzw8tSdTnpLpZuv3Wv9BExTeU298dgZJxmAesLT7gp8OeTkfEhx1LDDq6dM+8jx9ihzjqLFukko4HuUJFnkYB7e5D/uF/FWa46FKV+nR+NTi2uEZo3CfX8eMFDolSqg3Quj4sRgT6pDbY3T45I8GYpy0cYGdELDvYD7umHTynxYO5BgG3hTG0/VwD52ZEmeCHLnSqTnpnr3qjhcGwcugc7y2dh4RC9YYfhvIK7j94cQVHmVK3W6clJF/OGDKFuCcelYUvjwzd6hNE6C/0c564mgRD3gYEn/git0y0ZHbWLTN1SFQJ8YLIUQT+LM9GOQ8qNUv6dWiPXBJ+J7sXsluDZxgyLm9Nd4EFiDx5Xymn+mzJ4kp3EGenxky3rUjKRTerqLRFwHfJW4n2T6leIpa4sKt1XMln8jv38lI2tOhfyhtyAeP5fNTxtKyRh4+phbB3Xmfa3QN40YuZ2ZUweMUh1bEnzTTLKlulWn3sPrEvINbf9tOncZ0Vq/sQg/oVmuWfs7eHNHtfGEBHYx7p/Xz2FjLhdelDcrxOkra/kqePnVH/68IuTs0pH1x3t4NB2srasjqbL1T6DuP12snkE9Dqe3wY/ByS7BTEXMxFioAzhU6PCb41UnVRSTenQqMoJgTjhJbx4nZtmz20L/Fhvytoh4jdDKt8yTXXDrDKMn1S1HwngQJJVbVQyCWsNZBD1/W7//AdTJ6b4y/ZyYvKgDRTu+s6Vfi+KI7Fat3GV0wc3c/MC92MK8nTfWQ0UBMRT/UvBLaSqW02FCqEkau1ku6Id7sc+fSTDziR06Ozg3E0hFqXTJrOPd6bz7dh3ZIsPuCWOcOi1NXCO2rgwwfpoWJu0fdGmLXxbrdvhxXTqz2Z3oU0Vw4st2ROoJLZsgYCHnn45CgGIY/ocfd7TKCT81q16bbATexvIrkBNIeHQrlijdzOw5jzefVH4DUg7Chq4QHlvyDDmt3VJhA3+f7Md0NBj08S/eN+5M+X7lvOfNp+IwUO/BMbAXIFVluZFhI0iaWAtWPwQyM5T8WbCTVAAvEoKZYux0U9LvkBeBDNPdVu+KXGXgmqVDxwIaAYtf0e+brRhzYxitnMNP7T++scyLW/7Dhr056dLCkpscss6RsX3/8D/OFR4+2yrxlwbiRittM3q5mq13mtbGHfqfdaOxsuSJjt8izvfeJJTdAEDwfTH50NIvr3XXdJHSXcIvXNWs+MZbzlzJdLT1/caCputgyrZXQu9411EGJu5cAIF2Uk7EkF/X30areFJ8gM6mLHNbkPZWptOqhOB+sT3iQHIYKQ/CDNqhT7i8e35SUjKLvhDq3OAV4s1vS5r59x4q1YU8RP7jforjuC+8H7TXTCxiEq3bo3xH++utE+u8vvvDEHSJU7H0Jcq4HduWPvHc4fhPv7knoPYl+KtI8kJ0c3lugP1TpGj2OwvXCz0nfkXOYLwR0RgoH5h4j9D0rJBqruMRnVD2MAZAlxQMgsXgAZBKdre699jWQbuGaVy0xsKrhKPCVRO/0yLsCPYwkx8Q7bwNRQm+shRoRr4/55n3bgKsnUN2rNnQFhrzwGAzWXisELI7VsCVMqgUzi15kmXxOlp6SROeTKgkTja56aN+jrQ8MIA72K/HMLa4wdR/cyPuf7SgZlc/wNdzRx/HoTy/e/I8FkDe7ZFRn5bNLkHKwzhBoF+6botl4eLvCdYFcWzF68naXLzYYErI0gd9gGHZZFBm9yfsRY4BA6ZRKZU19//qrP46WkJrjOg/cpWQckp4odaD7fwXdDxXx2vsAgnvZB/mLh1PZ+8X7C/mDPQx/reFTXFAOv/6bLhXx6AIU+KTdfq+9CARTxBESD7ef4FtNSLruwvvFaG80GuH/J/3/CHB1akztcrn3tibGbp7beF5BT++Eg/gJ6kNP5veelqTc4TzDtwP0JKC4rsYqrt8Sc6QC6yYVq6KsOlK8vUKE4OwoQE4AcaNmz1stvhU7iSLkTXKbbnfb9oKrO23f694BjxHT+xEoBSW9KKyuWL8z9sRcRfnN9t5le+7sjnQx6yaDoV3UxpqmOAYWvcLN8g8om94JjeRIwV+4TAAiaxfR4QfYQ3ILBb4/kDnRSaANHe/VuE0AGozODSY3GP0Qiys2H5qxOJrKKmzCiefLp3DQtBNv4fiWpofGIUD2xLPrupsyFQH+QCS7fiCKDGGpr9MiThJZrObSP2GGtZsHS1B04HU9/Cxmp4CyivCeFax/1qqZ85mqaj7oxAU7J3078VWsBc41EXdtTzIx0XiDsydohT2Zz55IDngyn0Tnq6PFaI39NeXkjuFDy5lvzrSlfWtrrysmZj6ebobp9EGGoaJ+pKlZmK3f3/QwJnAdWxBja9gvVlBGV4p1JRknxReAujbCjshw35EmpRjOE3pknk5GaZG4MgzbSX0esv4E8TfD6Z+IU8hDIv/J7erK1skt6zvHj2yXkB9pra+T8njD+kbzr2kYCc2j608jMsL+egKf1qLcb39Qe2BcNBRuKW0F73ovlGtne3QXjjyLrjXOF/sn8+HwIez8zpLej2Lp1uZ/BF+7EXgwb7coPJjBD1DgQUzeYvBgTj8yBsd5z6L/ySx4lPYPaPpBvK83/Dn439orOnUSdILtrDUOFokrFQ79DT/2ig6TS+35wV+xzgkHjVJcbMfg/J++vL3NigbHgNs2I35Ds7il4U6gh/YIJIAx0VMJKbmjDAx5sPraQ66ecOepalSeUD1OqO8lHW8dfcOPt8o6nZzStmMeiT3c1jvue1Rott5IJ9qfgc/6TC4Yw8six1ZNrd1poCBXslsw8tH4b9odp22SrXY5bWxE3NtAXFFLr8FSj/dHl41vXcScZdrJovam7WVE04C8Ff8Ag4yERVp7nzas2eCaRo5W+GFVZ8UsV4x7LtA/4xK4hFG6BVw/AtQurxmwJN+y8d0XRkfedyuqbk1uVcAXCtZABYxHIb8oGn2tgcYdLrVH1uXzs38P7XqhKu4+ggL4Lit6mdBDZFGNWsC9nmtoplhpTj3yvWIFVsVEuizdpo0gRUKEazYggMl3zB24ra2Jbr6oxwSkx2C5O7bBWyn9v+c+dz8Snw0d78O2z1svbpI1k/p0wB9oVudE+OOu6NbVT+0rKO8Z2JBK6deOI3C/cPRzWvoKx10OCN0E4uVyx7O2AI4PlEd/S8tX8Fec7acnbtus797GL69evb74cPWSnrlNqsUGONQ0IVHUmjultE1kIP7u6u0P7z5EW/tJVxNKiuJDMGIvSb9SYmIHGiJGuNuNnQgcIYWiG3wcA4QOeZmKZfxoSFNo44NXh8ZlUae39sshFCooW8MXhR2N0SEEQES7Vom3tc6K68B/6g+H7lBFqJpG4UBH3B06eIMfVhUehTiBEsfu/mufb2427bvGeFmwuYWA7ii0PvXXneV1HiBTT4hecIUFXOF+CvMS77/ouXPvLcZzwzh8+NL79u2PJEYqVu1y+YoBtObaDsDLRF4cbunDBneFSPvAJWHRiNOhSio9I3Fut2RtH2BL52eHm/ovxkouqtrNCC9poJjcdhDO2C0QM8X3p2W4A10hg0cBN4xsezqi/WkDrIKnigVi+A59zEcaxgPlUECIaumcA2Dy4ePKYpy0yamX6i+gnw+c9sxtcY8ZCQMnoBQUdo2nvuut1X1KEUUg4eqmgz38945v1VG3xfExjJyJ0yWHWqW4OPNzRVoVOmhVFA1dUa1GAPpIeT78kvUNZFSIpIlWGuHpSXbzXKETUDtiOGjPSEok1BDe0NSJ20x7IMXxrG8u3l/JyKNYlBFxJKqosQnZFuLHANvno+3oE3OnnHDC2A3auLXSrXlr5bJbUOyWTMsaqqOffJSudymY9ZSg9hHFoeQ8XfFTNwBXVrg5y/CZhZgIHnQCIEIjgscKunDNfXeutiNoAcAsXzDeut4bwE5odCIzkKiroLHQU2wqj7r9iPoBZxFQZUFuYTQkquQt09z7Zom3nAhL3kHcIGB529jwXood43TrndNHvVnFdGkTLI2A/psfXl69ft813bSI2rbsnI8HqGBAoYbGw3XL3IGgG9G2dWeZJJk7pNcczHZE1CsTjkw+XK4DrVI8c5O1l0g4jegImbp8ukMiUYdJH+ch6SNUwcasPSF1qlbv10kRRvLXT7Acxcvdtgx0qW4fDycsI4T1Q0ePtGOFtq/aBJyZgTxzvH09XtQfLQIcwEQvH0FJ3w5jb29ldfX2KHrdCKJH4Nip5DdA1NoAeDCOevnT0dNvtLVOAbd+Auu48OOcQUYsd59baNi7Xya2wkRD+l6Ye+tM71LvfljPFeonnCS0gtN8/QoTN3RffDT3z9AbE8fjo90x0mL7i4dM9sdG9wRuH0eOh2z+Wux4COeD0bt3HMcz55I20t1ZY84Yy1ETyyuHzenSmSqOVnWuPtqqxeenNafkeF+1yi8Wf5QRLkKk9z3MRuaMVIb8yREdSRUR+p40tKySTv2J+5lMZVGW69Uh21BORM66u6o9nSFuIQQIK1AiPOSVE8YAFnOHzoSOd0VbGrmOPfsytEMCdY8/3+sjakQW9vps+IqvqZFCsoqbZrj3gcz4Ph+XrkH6f8/1IEEurpW709A2D0+rGngvXwf6LemmqspDDkFFNbr2pO3aE1u3vRAeLgyUoV+kzg7AbI9JXYxjutU0jtGZEce+vLmInkH6X93jzPL/nQAA'}, '15_expanded_frozen_benchmark.py': {'sha256': 'f3a4b03ce5951b4ed2da944f83ec415a588ef8dbd14cedc60c6f02e911415f42', 'payload': 'H4sIALS4K2oC/819bXPbSHLwd/6KCa5SBvdIWJLXu3t8wqvS2XKydX4r23tVCcNCQHJIYgUCPACUJWv539Pd8z4YkpKylyeuXUkEemZ6enr6bXqaf/in57umfj7Ly+e8vGHbu3ZdlS96URS9ycusYPx2m5ULvhgusjZjM17O15usvmZf83bNMjavNtuCt7y4Y8u6+sZL9u7q0xc2y+bXs6rkSa/3ZZ032Auv8w0vW7ZreMOyomDZTZYX2azgrObzql7k5aph8W7L2oo1/Aa6gjZsXmRN02fLqmbtmveW+Q0HLJp22Oy226pu+YKteLnLS+gmW2VNwn5u2TXn20ZgIrAaQI8Fn7cwcsn4ZsYXOFyvyO5gCJrKTVbkMMW8KhnOdACAC5pdViO+rIARsppt62rGRQuYHlvmt3wx6NV8tSuyOv8G2JR8VwPdCO98mfM6YR/qfEXENBMdsLJqASjfwp8LvkT8l1WxaBIkfa8HaG9Ymi537a7macryDU4WkIJmhGXT66ln9QqQbLj6vJqrv9ZZsy7ymfoofsGDZNfmhXr6a1OV6u9N1q7V3zUQoNqoT9/y7TIvuMBrXhVITMRCIfaq2pUtr2ku2a5oF/m8FcBb6BPGVIAfcQh60d5tgRLq+WV517PQ2BYVYbq9w79Y1rBt0ar35W6zvcNn5VY9QiaFBwi3UM8ans2qusSHDdBLPQVMFzgXer5Uj9uqnuu5/1rNkG6EZnNdwMKXyYIjN1RNTjyiZvPq0oUSbJJuqgUvFNDbapU3bT7/xFfASw00d9tssjLHtVfwXz6/v/IgeFvnc4vY5XKH/bzL4Pnt67zZAicPYF3k83RDL7xOEKdU7ANrCp/bGhgKGHXx1zeAhNtmm285TknPV372oGoO+2KOczML+jab8eKqnMOowBafW1ygevF5nhW87vV6ny7fv/7wLv18dfWajdn3F713H15fvU3/evXv8DHa8LpN//RyE8nH7y/fXdHzYTbcPseNPbw5H/7p5buo9/bn91eXn9JX6d8u3/5y9Rmg4rPk7Ox8wOAX/YQf58lZv/dvV5ev01cf3r/5+V8B6r7H4F9UZhsejVhEOzm1NnK65tkiGgiodb5Y8DJd5BuA/eF7+XRRV9tq18Kjs+SlfPaV56t1mwK3ZHf04ly+KJAeabOpqnYNZHLf1fDxBR+qjssqb3jatAuCOoO+90Ay2FqsqLIFcteu4DEyMaI/Yk0LJBZPU/2kP6LOcP/BdHHfxSBQoE2a9hOUYASqe+kTdL4kwYSNEn4LbNvEsh/8V2eAF/sEWx1k+VVdV3W8jD7xv+/yGiTfmhcosUHcb3JihRG7x472kei72fI5YOKKoQSfpshOAreimpN8i63pDAgfjSD1A6O8RwkMegE/J0gYMTg+PoEySCvYbjhRbCYR97AV43fxlXgRxjhyjD/MBCUiQDyYllwo8UvA1Bwkeil7h0V99TcYwl7V6Pw8nddV04BGSrW+BUEYDVjUfQHYfv7yy+t/7/RykVY3vF7mbQsrAby0W9zJTjrPoY+/XH7GDfbqbwn+JZmN1EoK+gWYgA3/rDVN8h4Wpdlmc65YDB7W0FwDXNarHWr7j/QmXvBmXudbXNdxmi6qObCg1TLJFgschprE0XB4na1WBSeTY1jDhgGsa8ll4y/1jh9urNcdeoGtud21w0Vey22M/6R+GkfPxSDPv1b1NVDiOYmcJarpVNk8stlRVEEXDcnwABxBo/FxXrZaC45fHm27yct8A+1BAs+vn9RBdnuqgx+PdsBv58VuwWkG0DwjzTCOsu2WlzB93ctkerSbhq/wz2YI20ggE8Tl+4d0Ar9Bjy30bJbA1lYfL85QlB/pZpa18/WwAREexOH8aGOU+UO+rebrMDVfnJ1uDkIkh93Jn7CeYOXO+ZCsQms1GrBNeNoC30cPaK2t26NdSDEke7I3utz7aBamCzC0YhSKI9IfA7SSd6BdwFgjgWBELSkMtJXLNtlcw5aLxYeG9uuAkS5Jq2tr+5IRTe0q4LY4+gr4crQXAPdxtGuXw5+iPhppa9iMhSXSEbWEUCN0BhJgwHLYtGU7vsB+GrSbs2ae5+M3WdFwNS9h/+gtnoqtI+RGXX1tRmDpN+0E7dcJqVX4MZ0KQQB7PaW9PmK4qEJNiD0s+7FfiL3ZfSH33EJ1RePhKIMeEbXdgUc1IVCDRQgpiZV5CkPAs6mg1Kqudlu+GJ3qBMS2ZbPHCNN38NS2Ev5D7ZCUVb0BdwmNJLJqYmM/4D/01fAJ4ONNlkD2PQUE9EYYIrtuDQo+PAiATSISVNM+6W6re9PcmvrEajFNhEzDXgSmc/RYGmdyCArrAcyYt3zTuDPCl0BifIEjN+R7xqFxEwKK+wPn5TW/GxfZZrbIqJMRi4dqpMn5tC+6npxNwYD4CiqzbzXvW3QD83uVz8gwmWiA2ELuKNYehjbRzbTZn8ceWxPc1Bk/QQLEj56UNuIQTPXVZ/9i7a1jpptD0WX0oSzu2L3T1V6EAUAm3ICHBwYez5qWRV7De3d+eyZ+J8KPhU15L9hjH3mLIMSH2BQWI7tUVshMRnpWUzOtqgb7kHqQPETtBl0GIdpFhFmaL6KpWS6FxQT7xh0s+5yMXKkztWW9ajSQrC8F4myXF4uUYg+xM8XRafFD8sp7FwSV08ed7jAuPOiwa0pQuUE4ITFvs6stPFQLi0dJXuBA5TbJmqyus7vYFga+8JmCfiZVXc1+hRGVMV+AjUxmreciGy4sU4Jqxi/NZm3Wu+Wy4ELvWbyM0RQwt7OWjy3f17YwaQ2QPGIWWVGkLQetKdbSvABnf87XAEzIwRy/cXALYtwGOJt+X/Ml9piiVrwdsBi6yUv0HeQD6pv+7hPTAq9ymCc301MkSOiP2Bp3IGkshrI8RDUIYovITGgAQXH6E4dyMZmaxoDR8YYaZdNodpcKuf1UPecxlELOVSlylOMqBf9ZcUS9YJow7iPF8JY6USzfd0efA/dgr7wxUsMBcFBEmTDovLbECyA8YjQTI1zcBn3nk5mTID4gYTiL/TPJcoPiobaKXAZy4nc8ddsSzcCTbrFZZ0K27LBJKtlbLqdh60MYSoUkJvNP485sDxCGInagBSLCMhoJbMG9hvbwyfQCj5BvEQJ+7XUHGBOXqqThrVl6o5QXrnmitySoJ2/1bAYmEBKJ+55PG3o3sECwAc6kYxHsbdsArS13AQC3iZz5NMmbRd78WoH5GdNzpIC3lBhNP9qGSHSoEXV4som3aYKRH5Tf7N5w7x6D5EL7o6lwna14Erly4STN3GGBXPcHdI21MshoigFGXdY+YfkYQ6Y7n3saZc+KCoRpRgImiTqtzRQ9NaO22xEGQ2h3HzRqb+MH5T2IyRntdW9rfbvv39UC2CsDU55IxN78+kh59U53d08vR+xcHDQRO5B74Uxi72u7U+sUXd3w+k6foZnzH7bZwerMONESXvBb8NTBlq3KOU98m1Pab0RoabRJ6cJTOjuAPmLpc4pPHScWHHblLj7CvBPgIrSTbnmdSippV1a+TGXIZsREsKbnBQe6vNDsNrHlcJDY1p6Vv+geQ1F4hI07vX4XwNX2NhRx+vbeE709xue4UmL73u1jr07zlnjAJLwSPeS+s6gaRWbY0Tf4NV2QLLovHQYHpaF7McTC2d13KbE/ERL/peR/39G5JVIX4/hyse9NF5G3Essa48AYIgeRXOO0Y48jYFFe/S35fPnu49ur9NPllyvLOPUnZhC0jywETfCJLeqPH1Mcjvtn8zWsHM7x8DEF9V8uK2TTZYJ/xeb4QQ7uKUOASZoMT8LR0iAJ48zagSYTBRpI6tnMKB49QqF1DaDo55KEA9iMZq5ifjBPD9M9+7dvAxYFerm3UNwz8Tthhvfd+WE/pK2DXXnT091FnmUlRNsmu+YpHbPH81TGG0mskFBRB5AjWzKqh4YcE6frOGrw2DHyjyGdIAsBdtCPzDl+1LWpuye7YT39aiynMgi+pjFScXI4jmZZkYEaWERhYPDwU9gU9fji5dlZGKSpCtA746iYLTEK3FX+nrFvPk579lLIWCllSYiJmeDyCB1PpCU41wPL6+4+VwFQ//kma/CxUTsGwA2FUgBdHgOE9JoXXBBzmAlfEpWQljgi3YNMhnLFYzOZpFlnWz45m1o7DhvIVcMm3jGzuzNFVsg4wLv9LlyyzNsumxhkJoTmQNBHG8wBj04QfHIUrjt+NsvBPsxJbgt8tjVHihLi2SMwI6t8emQ4lD1pAyYP0oZOKdVYlKMgkhriE/MSo3Rn78wlQBxQv8BQuzL/+47HMlrRP0YcoXfdbuVDHVASk8hWsN1XaH7ZpwePQe5BMxSdH4MKTeDx9LbneACL9HenNhrvvnMr8ySAx8BZFrzWff8K3h2UpkglMsfSbD7fAeaYg2HYcBLpx9NjjTfZvK7S5bnXWD8+1FjQyhraWpIHjC2grcGd5odG3/t0bYKBKRmFRHvzEAs8YJoPntBD6fbQxRlmsybG/LSkqFbnZ1q+9gOgAdbp+z466QcriYRI82d6jK57eR1Nu365VCr3330HpAXDieBG1Fb7nXbHR23u6C2pI5OQtczyAtwe94QWTWuBlNgYeKhCulA+fUVPyP0VSlsxUaPkEvqND3T2QqqaFHFQSTs22H3PC1EBYA7WddwVUZ78mBwJNYjofC8UfdOBmYNhLJUzpQyq1LazJHEOGS14cEPgXB3dEh0M2Mg9RIQXs7ykj1L8DfAMDeTiql2Prd76SdbgWQOKS1rHFxcm82qbZOVdrDods7PjsYboUgQdMaBAxFBBHxrLYyQU0VIws+cstlACJ00Mqew+TLlMF8AmdT7btdqqfWTkwMob8AMC5ACg/bFIXmdt9gY/WsEYN4ZCcTQR0TBBUpeFIpkAM2ITnKZ7PNZ3A+3yncVVezvLp2jBPlvtakyMW2EWyTj+04B9n7yUnmtTNsksq5FEBmNMVBrTpEyvt2OBuXlyN1Z4mmfrHe/AFXwF0lukLQwsp7jgbQvQC863bmYS4HxX5Jv4DE2z25gQmaixYP5/ZOcGss1bzA+7UjGpqpucTO4/IWWa3YKbAxKllunH4H3YXWLKI8inatfG5nGT3XAgInnPA7bY5uPzH8/M6zkwK48dntMJrMc3p2OMdF/rfdvxRHDqIlvyFIeKBFpgUT+n1iy6bzA5WCVZvcK1yG7zZnxusatoNcatLrwQOjyRsqHvHAfmqwHLEAci5m6GJGoMW/40YD9KpgwnBBtUBepWEpwAkDbfWA4vB09c3s5ux5nVdL7JtuPoLxhsslhWcEequeOFdSA6r4qqhi1jM7RAO7tNGt6mgiPp5z+Ap0htHkvwsWKjjxVZdNxqu5uGbARPSRKpDM25Sq9rAaL+CjjrUu8/zGR8mF36INPXBfIssqyZg4jCRK2JyO8SP8UKO7+C2juhPJTZXTyRs1aznALHN+IkQ6Vt6TbLvG4UK0hZvNtssvrOoTsuhO5e9mp1Aq5b7J9P6jmP4yBNMR09KyNP5yCkoqjT0JC529DCpObI+zRVZ1InNRACuFtU0kGbiIPuG3uaNgCmEGOoqIo8ETWO3iE4e3Pe0TVPGjxk26vBm+7gly6vWnpIJINFdLlHpOUPKasC1OYNLyJbJUpIWAP7WJ6cjeiwQtSp1dIHNRa61WhV54s4K7brbHyWXJjnQnfH/wjdWOyaVkuy3zMQR7O0TnU6UtDISrJYpeEv73nsQCH0enJH7PyYyYClOhYh9xUsR2rQN4zkx5mmg4Oq1nYjpCiQgV26gjXuxHkx5obbs2zAFtyIvFEpuUFt/6ouM9mCJI4+vrqEPQw/4zKluz8lJbRe+L3ZY9v+qHcW2A4/v7/yxCpe9el64+5wgdAXB/uT3+bt3fiH7lvwB9pxtJ1nUShOk9XoLqQY/R9H2a6tAlDHk5W0LDtGBkvkHSTIL+8uP3qjIzMl+PzxRCnTEh27WVU3IaqAM0YuDW7X/50ZO1acCLfadtz5gL0YMC3iz38csJf26ZjM2pL3XzDrofHytCzOtePWR32rrn9FS3ELrpMYYzIasLNQgOrOATkPgUhP7XA8cR9QhOhQNfMM88xcxXLIsZLO1a3HOuBd+ZZNyLmiIcc/eScnSpL/9NJ7jsYvV8lo7jvppsmMqDG7CKl5q7Vl67r50j6MdLUm08Mgd10QO4Xpgg7j7EYraKR0E0an8PTUjVD5w9gtwFjZVDc8VmY7QF1Mla6bzarbtK3SDNRmVQMnJ2cXwCD9ASuq+TjabdGbLPiyjfSOgF2wFZToWaks1v1howtAHS8xoG3O4QZSLd/TL5mh+zSFO2CEfA6Y8wa8c2wadbUwaSe6yYi6xb7Y2NEsNu/3w/GxQxHuqMmLdbVDJz+SR57uZhBa1ECJsKkjhQYKz35I/kaL7AZ803SG198oZ+3gKC7ko0ba21YL3ohJVZBKDLTO8TYKee5P88SOSNQLW6K+QKv5J0umiggHClGDw3G5KVpMIgF/Z6cg0BpjJiS5MOxegj7DT8+mGHOxksdpy5xNk66Ek8EaunPkx71NIAeWAHix8QGEsbyM7umPvYgGRmHR9uPLg/LpqYihXR9EKy/BC7srQPoOh9FxlPFY69EIn/+PKHnomOEfOPAhD+gUtQ6jpBbOKJXoLSzFCA//84VKRl1kDSaBGO8nslqfO62Vx/XAHnA/ZZSUjX2NLDIJHSadrysiik3FoOtkz0b5ba/wrusQzL262t4x4rMu8gpY+4shEBEaRb3kjiYV2LICowZFBphgINOK8UVAUUVv8II4sys9GJXEMrqYaxL9KFcv+n29wK91DjK/5uh1xR0hSfOq0R3s3tJ9UuIf3hLOisKGRvk80GdKIp1n5AhMFYAlV3XUcRxdRGRobqSLR/iSvq1aFVV5YtIg7S3nxkn0ByYKm2h/X9gdQ7I7PuEpwV/03WpLddp//0H18Y7KPLxaVzle9wwDU79gP8FuarrlUhL2Zc3t8iiLigvrbFcua86/cSunKsrAJsI9CENrk4PLvBa0lObZruFYJwUjviL9q8L9U7BmXX3F4iSV3d2Ml8B+LRPlONRJkrCuzi+SI9PXxMOVh012AHQZDdkXXMTQKcOI3dsLvPfafaz5TV7t7NoxI3bx0urAa/BzOa85IOP1y4bQam+fbkRWs/j+/OyMfcdip8lzHGgINuwoOV/u/7nvjfSm2oH5ibst8TNgk7PlXt5gtlIp9eA2naAfLGZDQw6/4uEeSY3/xzglDZtcYTDXA2nCblcfSm7HuExjdZQDm6TJF8Bq2XzNQApxcTvo0Br/JrbCb6HaNew3B3I4HOL/I/V06l2L6yRzd9LmaZfqFHIY+x5b7mH0e32pCT5F1p7u3ApxA+qRp0eRZ2mzNuwKJQS4z36svdPkPGF0Qi/WppCpfqgBZK6fXzhoqC8Hii3k5jdG6JK9EkGyhH3Wp/5UDym0dEMKaLKNDMMmPn4XCa36sqOXOhWIgF9IKDRo4WokPfRkFPC/fvzhJ5TCP3yPP1/+F6glUeWEapwwcXjOqLyJKK0i7GCvM13mRMLIsBOj9FgsdEIOqvbyGBU8oVInyalleV/JCdJte7a+w2hYhsoHebr5yjnl9sJTko4LNXmtmz1UMcSAUhja1Hyxm8NqIHidN9esWjLwaSsqa5OxYje/vqMDQQzPiztNp7AFvvsgFCn7sGuHH5ZDurHxiTe7om1Otf6NvTKLiB/yLVNmJHz+QvnjnQeKY+CBN1fxvq22wxfOJvY2svcj6uaH2te8sTYHzgb3uDQaTmxxL6Metjn2Q/tddDV5hrLz2XTyTE3u2XSUvFjuO1MCR0I1IUH61DbqbOQxbYCO6Yu0M5qf6f8keTVkV1ld0N4FA3RD+9ousCZJDrztqEQsF/Tih7Nub2hlIOPRtlM7gbz6BgxGThbHIoeOUL3kWEYhn6OVAXqn8bnItjOMogErgO5Szdd4rL14yM6grXAJGH0wFWnYZ3FqdKI5quHPjrgF++dO5jrd0w1Pz8zs7wNdkFEvBAmeSxgjKNPeUIcPjO07eWbCGxYfJNgVMD+ywwPGtGT/Y0a1XcqTY77Tkxtaw62y7UNGUqOkAH90pNAaX1nlSE6uqExcUEuK1/CFKzF5Rg+eTUOT+6wjYoKj7WYmWib3Z2hUinUN/yJiXd0+3FhYuJ+umsIEYbTvUEqC+hviJW1m0KG1R1WTtbZpBRoHt5WvTlFxkxW3qGiromoDBY+rKqSCKfuivK/mQTuQTLuf0fva1lwkb5xqRqoyF1qS6iUxUe1OlmhkwCw1eTLCNRg5cotMJt8wYs0GFSSKyLr0aif+vFSqHHP4tCOFQzbtIESsEvphs6oFr73keI9PoFrugEI1CcAFrEvd5O0dklt7Jl43fq3GDJYRGLOtjGe2wcA0TUuImIRdsjXYR50JKg09p/qNeO9vtcP0n5YLsb5hL86d4pAzsLdapLPX07au2moO/twKpiDmtcGUvwVfcrDvsS4IkCXfoJlVzVogFad4Z4BM8vQoOb3cHDTEfI28nBVNRS4lrQEYhcNqOaTwp8kzb4jEZG7SqnrDIvQQC6XB+s5tGxXc1fm1OGBKBFHzFkT4Up1WzKtt3rXelC7KW80aOPSulGqIbXezAgZC7q4FOYhbkwNGDd0oE5GWFvgojv6zjBK8cxyTGu8HShTpYmlzvD0ci6pjKShTEZoR15YwUmOX4zNQ4GxGbtmxVGyYlCqSiXhP8i3fRgbBXQnYXMfy/lqgrJIs15n8R759A79lgIlKLOlXP39MX1+9eXv55eo1lVmSi+zegFGX9FybzcX+09XHD5++JBvfo3Kh5PWDozAi7+U4DC2fDdK9f41IJ3lDBQX9e4EiTkYzFescvrGFXQwOvklqXpB0T9vKWnDM7U2xROhtHEhLd1PQeWEhivWyAnhSCQ+sVGoKRIjRV0U1i6Pvon6gkaQBrfERGjyCFhod6GpwEuJJtAleeEam1ZcCYQd60UGMv+B5jV27zLobSgVdk/lukSEZdLHhE+Uso6uSShJn7K9UHpB9xPgQrMOX79m/fvxF51RTvkDSgPyOrXQB8bLcJkffC8xAn+0yrMcaAqFDcYzNr/kGz93odODrGpYIo+coc7yNIW/LUkzKJjrsJrwMKEkj96BsEd6fKvSNGzEIqDapqpIJuzHcodio/sAPLQ9nofDQJgaZh7SwLw1SxTFZ1jgVhSE7d7sMgSUAGjgplqf0yeyWihpQiQV9U+BYATqFkdkgNJ6uIuU9d4tY+S+dSlDeS1k3jXoNlSCyi0K5hUlE+MO5uhWZCC1W0g2qsRs7oTTSrpnO27/37ilhQZ+jF0NU0Ydpp8aJjDoG60h0qpw4SDmh3GjEQhHebgPrkr3XxnpjNSMuBUhT2XhgVa6ga9swXaCwLN+Nl1MvXv4Qd+sfejnFYmm8VBdMQL7md41Xj0qkNPhFEi2lmojj/VhbOeLpmt8u8hUWnOhPRucXUwvnkAygNxH8pabVU3e+lMwwbRGcKl+6CY4hOKvGpVP3QlSwxXLUGU6sEDnNFpn0nuzZ988IF2+HHFz2kwtMAFSLU4xu768DdTvsOQyOIPuwsYOoq2RHvE9mcl2BVMvCznoR7y/Bbid3U0itUWTLMBvScLUKxoysfX7vF1QMFlL09uY+gK84Q7LCXIDQ404DA92Jyh/RwC0OIvUCfdsBcRN4AUDjlrgplLaq65YPvEdmZ4eX1+FwbwmpiG2Kh9JBvjJ42MSykqbGNIGJvVGmMk+2xhH1hWYBJj5KECVwzWt1k8hKhKqDiVCx033fLcYnGyZ2/mQH2r6WNhZFImUz+TSVCyQL7VfgwdPtu+WuKKzkXeu228Du1LI9MQ06K61LMVQC0FzLs0lLcTBnrLTIr3lssBjI/vompTp1KxJQXM195IYo5f7S5+FY+GBqlYdxzYpU5zDZQ7g1/8S3Ctg5o6TSLRNY7IgluLlXdu6CXVKKUpmev7SSOOhyKYxqXzVF+IHhHTtPimoGqCvbrN1RgHUcqC7RzQQP5A55z/SI7mNCKpTGEyS5TDhnfxyz82BNQMJ55CGjllgda9yLyywjZwnoqvC+HyqQ6BZuYL85dRV6j6ktcaCuRLdygxo9lEfWTAJvzRjiOzFE3eVT6cTKxCG8AonC7iIcuewvdy1A+GLASyz2Vt94Jc9lgl7qc3SqTnQTMbMoNGmzv3WZDSo6N31I8YzDJT1EH9Z4PQdzUG50b2Eg8xTvBup+mb5IkZeprmf40FGDBUIeUkTkRAWQYx2GICxx7L6wvqDDS4ND1UcyWNRmP/RWlV5339tfNfJHe3O6FT6P3BJ/LMGOzjJ4HQwD6M7aGzFEsUH5TnwQEJodakzieTg7PFoGPG3NTOEC/JnK7Mj+saXBUMvvtkAHJvL4pVG6vysBBPkF59mXf70DbmsBAy/kVn+U5AihKaJKmFF4UjzTNZcUxVbH89YkotQ+kayy4C04X2Dhzbe7uB8uRKlzAXTBJGvaiRkQevF98AMy/PdSFElbUemAfqgVeczQ6CAXE1heCpcWv/wH2bpTOOpiGurdYjNoeJDpnqK7/O8pSrZt9CilfZj3HjQ+8bxofUx90rHgIfnVNcm0RRtMFAmaVzozf6R05T5YLUSa3uGOD9zFHh0URY/jTQ14vHSR0Oo6IoYeDI6vdUz/2OVut4UowBsawxRCdYcQpXQDLWilLQk+Yl2hfqiZl26vmh7PwjfNnaR51fhYJr1paqcs6KbOwxNIU7q5i3DopoOLrNPo0OWIEH+u5ir47H4nwwPOMIzoJzC+2bZ3KQXLYruuTN6Ab2q5q32qNNOn7AT1Vik6+e74OckHUTaJjsDdtHfKJ8JaOCLVOdGnFTIPzvnmBHm90alL5xw34rWGTuYlBm3MVKwLpoEsffE1blqLq8uZIycC+OB6aSEX9EhttKDQ7/eO1VZ7Ssm5Ixg80Gf2cHowPQ7f9TxZN+4htJmQElXxJxkR6t6xs9hrghw1pcLsuLCygttAlghSt1P3Pe8ag3/xS2uMvgcI5kQ6b25i+yjtuVAYqUjKaxJ4H4lvI3KqWDgDmPBBX3VpnEC3bwEqYhCib6tOrR7DDleZL2/yupJE0pgiIKAqH+vjS7Qiv9F9a9xw9pcNeP1RyRDYVamV/5GU229Rp+BN+Ar/OMCKkifGhz1+ufcd3hpboTjXdHfBlBxwo8yayd34nPxaFXMFxfpCEf2NDW1sol790DeJUCVR6/rq/1dp939R5Fj33RWcG7V016HvF1+l1aO7uWidB2ao7jTLokz9fqgDtfxBGzFsJxrXQ1QIHtnrHIbWx5kjPdeDkDtxQBtwaib2isnZTQ/0o+lzsDMNcaiLeVVjajBF9DSxxyyAxMEOwNFa0LfiBS8dn1AcE8kdBtFg64D9ug9UgXTksLf4J4WxwM4Xef8zqewdhyup7B68H2wtvoBBthHxfa/BobmYc3swX+scTUvV0YFifN99FzpkjtTpLm4A+acHQUoHj+RFwB0PyvWBogeqqeHC3weKdhz4urZuNsDB08Zjnngkpe4rWcELkPCqQHvwJHVPxxQiK5KpMgYOBjcjJ7DpgIejnZGqZcI5rkewmIk+Y5VXTYPlHsNH0HZm0HMr2UNm+zr9JFt9461vRrILsrkHKadHEbBWQnB4CK+OYEgphcyFwxaHOTG3kplHoUth0QMmIYbWSD55Dq4t86QZLA9YECcnoXk9PAVTYfCOItXBY7cEc5fRzNzALM/7k7Mp/GffIKYKD53aV4eOCJ3jZM/SdE0PB7vTczUn9Tq33p/vQeksWyjvSclp+dha70CljHBw7IFro7pK57v6hncx7l4nf1hmL8qfYzkq0pcYuME3r4aPnP7g+LGsUytSpqCPA4nWTnLHf5biUgP3+F2XddOXsHWpZdHSSqtSc9DfKOvkj+BXniPVQCc55HqmyfVsH7kNLmVuNd4apr/2lDeeL1lK33CepmhPRWmKia5pGqkipJj12vtvqXgQ+KODAAA='}}
for name, info in embedded_files.items():
    content = gzip.decompress(base64.b64decode(info['payload']))
    assert hashlib.sha256(content).hexdigest() == info['sha256']
    (RUNNER_DIR / name).write_bytes(content)

SCRIPT_PATH = RUNNER_DIR / '15_expanded_frozen_benchmark.py'
print('Files:', sorted(path.name for path in RUNNER_DIR.iterdir()))


## 4. Run The Final Benchmark

The procedure is intentionally conservative:

- MERT is completely frozen;
- all 13 hidden-state levels are extracted;
- the embedding layer and logistic-regression C are selected using
  recording-level validation macro F1 inside each fold;
- one fixed neural classifier is also evaluated:
  `768 -> 64 -> 5`, dropout 0.5, weight decay 0.1, label smoothing 0.1,
  learning rate 3e-4 and small embedding noise;
- class-weighted loss handles the 7/7/7/5/5 track counts;
- every original recording appears in the test set exactly once.

There is no further neural-head hyperparameter sweep.


In [ ]:
command = [
    sys.executable,
    str(SCRIPT_PATH),
    '--kaggle-data-root', str(KAGGLE_DATA_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--num-ragas', '5',
    '--minimum-tracks', '5',
    '--maximum-tracks', '7',
    '--exclude-raga', 'ragamalika',
    '--segments-per-track', '4',
    '--segment-seconds', '30',
    '--batch-size', '1',
    '--head-epochs', '30',
    '--head-patience', '5',
]

print('Running:', ' '.join(command), flush=True)
recent = []
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='')
    recent.append(line)
    if len(recent) > 300:
        recent.pop(0)
code = process.wait()
if code:
    raise RuntimeError(
        'Final benchmark stopped. Read the full log above.\n\n'
        'Last output lines:\n' + ''.join(recent)
    )
print('\nFinal expanded benchmark completed.')


## 5. Read The Final Report

The report contains the selected recording counts, fold definitions,
hyperparameters, test results, overfitting gap, confusion matrices and
embedding clusters.


In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

report_path = OUTPUT_DIR / 'REPORT.md'
metrics_dir = OUTPUT_DIR / 'metrics'
figures_dir = OUTPUT_DIR / 'figures'
assert report_path.exists()

display(Markdown(report_path.read_text(encoding='utf-8')))
display(pd.DataFrame(json.loads(
    (metrics_dir / 'overall_results.json').read_text()
)).T)
display(pd.read_csv(metrics_dir / 'fold_results.csv'))

for name in [
    'expanded_dataset_distribution.png',
    'expanded_layer_performance.png',
    'expanded_embedding_clusters.png',
    'expanded_linear_confusion.png',
    'expanded_head_confusion.png',
    'expanded_head_training_curves.png',
]:
    display(Image(filename=str(figures_dir / name)))


## 6. Download

Submit or discuss this ZIP as the final expanded-data result. It contains the
report, folds, selected tracks, metrics, out-of-fold predictions, figures and
small classifier checkpoints. Large audio and embedding caches are omitted.


In [ ]:
from IPython.display import FileLink, display

result_zip = OUTPUT_DIR / 'final_expanded_frozen_mert_report.zip'
assert result_zip.exists()
print('Result size:', round(result_zip.stat().st_size / 1e6, 2), 'MB')
display(FileLink(str(result_zip)))
